# 01_07_acq_dash — Jan–Aug 2026 (MPOS commission_monthly)

Clone of `01_07_acq_dash_june_debug.ipynb` for **2026-01 … 2026-08**.

## Purpose
- **Июль/август 2026**: только озеро (нет Excel-отчёта) → `no_excel_reference` в compare.
- Build `final_df` for dashboard (sections 01–10 + 10b products + **10m MPOS** `commission_monthly`).
- `commission_monthly` from `ods_alpha.scd1_mrc_pos_rent.n_amt` (unique `c_nmrc` → inn+agr_id; missing → 0).
- QC: TOP-10 deltas; row exact-match %; analytical memo Excel vs lake (включая **AUR**, **amortization**, **fin_result**: Excel `АУР`/`Амортизация`/`Фин.Рез.` vs lake `retl_cnt*1926` / amort / `chod-aur-amortization`).
- DRP upload to `sbx_da.tmp_shestopalov_acq_datamart_jan_jun` for Superset.

## Outputs
- `final_df_period_2026_01_2026_08_mpos.csv`
- `final_df_compare_2026_01_2026_08_mpos.xlsx`
- `row_exact_match_2026_01_2026_08_mpos.csv`
- `memo_excel_vs_final_df_2026_01_2026_08_mpos.md`
- checkpoints: `checkpoints_final_df_2026_01_2026_06_mpos` (reuse Jan–Jul + add August)

### Reload after new amortization source
- Set `force_recompute_final_df = True` (cell config) so monthly checkpoints are ignored/wiped.
- Amort lake table: `sandbox_ai.shestopalov_terminal_amortization_model_jan_aug` (build via `01_07_build_amortization_drp.ipynb` if needed).
- Then re-run month loop + DRP upload → `sbx_da.tmp_shestopalov_acq_datamart_jan_jun`.
- Dashboard KPI **Общий ЧОД** = `SUM(chod)` on datamart (see Superset checklist).

### Active retail points (`active_retl_cnt`)
- Считается в **05b** после merge `base_df` (не внутри тяжёлого section 05).
- Точка активна, если ≥1 её терминал с `n_amt_src > 1`; clamp к `retl_cnt`.
- Superset: активные ТТ = `SUM(active_retl_cnt)`; не путать с `SUM(active_terms)` (терминалы).
- QC SQL: `sources/sql/active_retl_cnt_qc.sql`.

### Kedr Общий ЧОД
- After period cell: enrich `kedr_obshiy_chod` / `kedr_obshiy_chod_contrib` from Kedr.v_detail_*.
- Then DRP upload; Superset сводная = MAX, сегменты ЧОД = SUM(contrib).

### Автопрогон (одно нажатие)

**Kernel → Restart & Run All** (или Run All).

Соберёт: amort-check → месяцы `final_df` → period → **Kedr Общий ЧОД** → (опц. Excel QC) → DRP upload.

Флаги в config-ячейке:
- `force_recompute_final_df = False` — досчитать только август с checkpoints
- `run_kedr_obshiy_chod_enrich = True` — Kedr на каждый ИНН×месяц
- `run_excel_qc = False` — Excel TOP-10 / row-match / memo (по умолчанию выкл., чтобы Run All не падал)
- `run_drp_upload = False` — для обзвона CSV; True → upload в `…_jan_jun`

### Kedr Общий ЧОД (lake snapshot)
- Сборка: [`01_07_build_kedr_obshiy_chod_drp.ipynb`](01_07_build_kedr_obshiy_chod_drp.ipynb) → `sandbox_ai.shestopalov_kedr_obshiy_chod_inn_month`.
- В этой тетрадке enrich **читает озеро**, не live `Kedr.v_detail_*`.
- Порядок: period CSV → build Kedr notebook → Run All этой тетрадки (enrich+DRP).


In [ ]:
import math
import re
import json
import threading
import time
from decimal import Decimal, InvalidOperation
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    return s or None


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return s if s else None


def to_decimal_or_none(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return v
    try:
        return Decimal(str(v).strip().replace(',', '.'))
    except (InvalidOperation, ValueError):
        return None


MONEY_QUANT = Decimal('0.01')


def to_money_decimal_2_or_none(v):
    d = to_decimal_or_none(v)
    if d is None:
        return None
    try:
        return d.quantize(MONEY_QUANT)
    except InvalidOperation:
        return d


def normalize_text_value(v):
    if pd.isna(v):
        return None
    s = str(v).replace('\xa0', ' ')
    s = re.sub(r'[\t\r\n]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s or None


def normalize_ssp_ocrm_core(v):
    txt = normalize_text_value(v)
    if txt is None:
        return None

    txt_norm = txt.lower().replace(' ', '')

    if txt_norm.startswith('дкб'):
        return 'ДКБ'
    if txt_norm.startswith('дмсб(ми') or txt_norm.startswith('дммб'):
        return 'ДМ'
    if txt_norm.startswith('дмсб') or txt_norm.startswith('дсб'):
        return 'ДМСБ'
    if txt_norm.startswith('дм'):
        return 'ДМ'
    return None


def normalize_filial_rf(v):
    s = normalize_text_value(v)
    if s is None:
        return None

    m_rf = re.search(r'(?i)рф', s)
    if m_rf:
        s = s[:m_rf.end()]

    s = re.sub(r'(?i)\b(ао|пао|оао|зао)\b.*$', '', s).strip(' ,;.-')
    if not s:
        return None

    s = s.lower()
    s = s[:1].upper() + s[1:]
    s = re.sub(r'(?i)рф', 'РФ', s)
    return s


def _norm_tariff_text_short(v):
    if pd.isna(v):
        return ''
    s = str(v).replace('\xa0', ' ')
    s = re.sub(r'[\t\r\n]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return s


def _is_zero_tariff_value(v):
    if pd.isna(v):
        return False
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    return s in {'0', '0.0', '0.00'}


def build_tariff_short(tariff_text, source='lake'):
    txt = _norm_tariff_text_short(tariff_text)

    has_akt = 'акт' in txt
    has_ind = 'индив' in txt
    has_std = 'стандарт' in txt
    has_promo = ('акцион' in txt) or ('меню' in txt) or ('сезон' in txt)
    is_zero_excel = (source == 'excel') and _is_zero_tariff_value(tariff_text)

    if has_akt:
        return 'По актам'
    if has_promo:
        return 'Акционный'
    if has_ind or is_zero_excel:
        return 'Индивидуальный'
    if has_std:
        return 'Стандарт'
    return 'Не сегментирован'


def first_non_empty_value(series):
    for raw_v in series.tolist():
        cleaned = normalize_text_value(raw_v)
        if cleaned is not None:
            return cleaned
    return None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


def safe_divergence_pct(delta, reference):
    if pd.isna(reference) or reference == 0:
        return np.nan
    return abs(delta) / abs(reference) * 100.0


EXCLUDED_PRODUCT_COLUMNS = [
    'acquiring',
]

PRODUCT_FLAG_COLUMNS = [
    'rko',
    'accounting',
    'business_cards',
    'credit',
    'dbo',
    'deposit',
    'insurance',
    'loyalty_program',
    'nmo',
    'nso',
    'pravocard',
    'salary_project',
    'self_inkass',
    'service_package',
    'sms_info',
]


def build_client_products_long_sql(month_end):
    """Active agreement products + rko on month_end; no OCRM / segments."""
    return f"""
WITH rko_dt AS (
  SELECT max(dt_part) AS dt_part
  FROM sandbox_ai.prod__kuznetsov_lle__acc_ul_trx_features_impala
  WHERE dt_part <= date '{month_end}'
),
rko_inns AS (
  SELECT DISTINCT
    regexp_replace(trim(cast(f.inn AS string)), '[^0-9]', '') AS inn
  FROM sandbox_ai.prod__kuznetsov_lle__acc_ul_trx_features_impala f
  CROSS JOIN rko_dt d
  WHERE f.dt_part = d.dt_part
    AND f.inn IS NOT NULL
),
current_agrs AS (
  SELECT
    regexp_replace(trim(cast(inn AS string)), '[^0-9]', '') AS inn,
    lower(trim(cast(target_name AS string))) AS product_name,
    1 AS has_product
  FROM sandbox_ai.le_nbo_scoring__targets_product_agreems
  WHERE c_date_begin <= '{month_end}'
    AND (c_date_close > '{month_end}' OR c_date_close IS NULL)
    AND inn IS NOT NULL
    AND target_name IS NOT NULL
  GROUP BY 1, 2
),
union_flags AS (
  SELECT inn, product_name, has_product
  FROM current_agrs

  UNION ALL

  SELECT inn, 'rko' AS product_name, 1 AS has_product
  FROM rko_inns
)
SELECT
  inn,
  product_name,
  max(has_product) AS has_product
FROM union_flags
WHERE inn IS NOT NULL
  AND inn <> ''
GROUP BY inn, product_name
"""


def pivot_client_products_wide(long_df, product_columns=None):
    cols = list(product_columns) if product_columns is not None else list(PRODUCT_FLAG_COLUMNS)
    empty = pd.DataFrame(columns=['inn'] + cols)
    if long_df is None or len(long_df) == 0:
        return empty

    df = long_df.copy()
    df['inn'] = df['inn'].map(normalize_inn)
    df['product_name'] = df['product_name'].astype(str).str.strip().str.lower()
    df['has_product'] = pd.to_numeric(df['has_product'], errors='coerce').fillna(0).clip(0, 1)
    df = df[
        df['inn'].notna()
        & df['product_name'].notna()
        & (df['product_name'] != '')
        & (df['product_name'] != 'nan')
    ]

    if df.empty:
        return empty

    excluded = set(EXCLUDED_PRODUCT_COLUMNS)
    df = df[~df['product_name'].isin(excluded)]
    known = set(cols)
    extra = sorted(set(df['product_name']) - known - excluded)
    wide_cols = cols + extra
    if df.empty:
        return empty

    wide = (
        df.pivot_table(
            index='inn',
            columns='product_name',
            values='has_product',
            aggfunc='max',
        )
        .fillna(0)
        .reset_index()
    )
    wide.columns.name = None

    for col in wide_cols:
        if col not in wide.columns:
            wide[col] = 0
        else:
            wide[col] = pd.to_numeric(wide[col], errors='coerce').fillna(0).astype(int)

    return wide[['inn'] + wide_cols].copy()


def merge_product_flags(base_df, products_wide):
    out = base_df.copy()
    flag_cols = (
        [c for c in products_wide.columns if c != 'inn' and c not in EXCLUDED_PRODUCT_COLUMNS]
        if products_wide is not None and len(products_wide)
        else list(PRODUCT_FLAG_COLUMNS)
    )

    drop_excluded = [c for c in EXCLUDED_PRODUCT_COLUMNS if c in out.columns]
    if drop_excluded:
        out = out.drop(columns=drop_excluded)

    if products_wide is None or len(products_wide) == 0:
        for col in PRODUCT_FLAG_COLUMNS:
            if col not in out.columns:
                out[col] = 0
        return out

    prod = products_wide.copy()
    prod['inn'] = prod['inn'].map(normalize_inn)
    prod = prod[prod['inn'].notna()].drop_duplicates(subset=['inn'], keep='first')
    drop_excl_prod = [c for c in EXCLUDED_PRODUCT_COLUMNS if c in prod.columns]
    if drop_excl_prod:
        prod = prod.drop(columns=drop_excl_prod)

    drop_existing = [c for c in flag_cols if c in out.columns]
    if drop_existing:
        out = out.drop(columns=drop_existing)

    out = out.merge(prod, on='inn', how='left')
    for col in flag_cols:
        if col not in out.columns:
            out[col] = 0
        else:
            out[col] = pd.to_numeric(out[col], errors='coerce').fillna(0).astype(int)
    return out


RECOMMENDABLE_PRODUCTS = [
    'rko',
    'business_cards',
    'dbo',
    'insurance',
    'pravocard',
    'salary_project',
    'self_inkass',
    'sms_info',
]

PRODUCT_LABELS_RU = {
    'rko': 'РКО',
    'accounting': 'Бухгалтерия',
    'business_cards': 'Бизнес-карта',
    'credit': 'Кредиты',
    'dbo': 'ДБО',
    'deposit': 'Депозиты',
    'insurance': 'Страхование',
    'loyalty_program': 'Программа лояльности',
    'nmo': 'МНО',
    'nso': 'НСО',
    'pravocard': 'Правокард',
    'salary_project': 'Зарплатный проект',
    'self_inkass': 'Самоинкассация',
    'service_package': 'Сервис пэкэдж',
    'sms_info': 'СМС-инфо',
}


def build_recommended_products(df, recommendable=None, labels=None):
    """Recommendable codes missing on the row -> comma-separated RU labels."""
    codes = list(recommendable) if recommendable is not None else list(RECOMMENDABLE_PRODUCTS)
    label_map = dict(labels) if labels is not None else dict(PRODUCT_LABELS_RU)

    if df is None or len(df) == 0:
        return pd.Series(dtype=object)

    def _row_rec(row):
        missing = []
        for code in codes:
            val = pd.to_numeric(row.get(code, 0), errors='coerce')
            if pd.isna(val) or int(val) != 1:
                missing.append(label_map.get(code, code))
        return ', '.join(missing)

    return df.apply(_row_rec, axis=1)


def build_mpos_commission_monthly_sql(month_start, month_end):
    """MPOS_RENT n_amt -> inn + agr_id (unique c_nmrc mapping only), SA agreements."""
    return f"""
with rent_base as (
  select
    cast(c_nmrc as string) as c_nmrc,
    cast(d_rent as date) as d_rent_dt,
    cast(n_amt as double) as n_amt_num,
    cast(ods_commit_ts as timestamp) as ods_commit_ts,
    cast(ods_insert_ts as timestamp) as ods_insert_ts,
    cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
    coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
  from ods_alpha.scd1_mrc_pos_rent
  where c_nmrc is not null
    and cast(d_rent as date) between cast('{month_start}' as date) and cast('{month_end}' as date)
),
rent_ranked as (
  select
    *,
    row_number() over (
      partition by c_nmrc, d_rent_dt
      order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
    ) as rn
  from rent_base
  where ods_deleted_flg not in ('1', 'Y', 'y')
),
rent_dedup as (
  select c_nmrc, d_rent_dt, n_amt_num
  from rent_ranked
  where rn = 1
),
terms_active as (
  select distinct
    cast(t.n_agr as string) as n_agr,
    cast(t.c_nmrc as string) as c_nmrc,
    cast(t.d_valid_from as date) as d_valid_from,
    cast(t.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agr_terms t
  where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and t.c_nmrc is not null
    and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
    and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
),
agreements_active as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agreements a
  where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
),
companies_active as (
  select distinct
    cast(c.n_cmp as string) as n_cmp,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
  from ods_alpha.scd1_companies c
  where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and c.c_inn is not null
),
mapped_raw as (
  select
    r.c_nmrc,
    r.d_rent_dt,
    r.n_amt_num,
    c.inn_key,
    cast(a.agr_id as string) as agr_id_key
  from rent_dedup r
  left join terms_active t
    on t.c_nmrc = r.c_nmrc
   and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
  left join agreements_active a
    on a.n_agr = t.n_agr
   and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
  left join companies_active c
    on c.n_cmp = a.n_cmp_client
),
map_stats as (
  select
    c_nmrc,
    d_rent_dt,
    n_amt_num,
    count(distinct case when inn_key is not null and agr_id_key is not null
      then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
  from mapped_raw
  group by c_nmrc, d_rent_dt, n_amt_num
),
mapped_unique as (
  select
    mr.c_nmrc,
    mr.d_rent_dt,
    mr.n_amt_num,
    max(mr.inn_key) as inn_key,
    max(mr.agr_id_key) as agr_id_key
  from mapped_raw mr
  join map_stats ms
    on ms.c_nmrc = mr.c_nmrc
   and ms.d_rent_dt = mr.d_rent_dt
   and ms.n_amt_num = mr.n_amt_num
  where ms.valid_key_cnt = 1
    and mr.inn_key is not null
    and mr.agr_id_key is not null
  group by mr.c_nmrc, mr.d_rent_dt, mr.n_amt_num
)
select
  inn_key as inn,
  agr_id_key as agr_id,
  sum(n_amt_num) as commission_monthly_mpos
from mapped_unique
group by inn_key, agr_id_key
"""


def merge_mpos_commission_monthly(base_df, mpos_df):
    """Overwrite commission_monthly from MPOS map; missing keys -> 0. No cascade mix."""
    out = base_df.copy()
    out['inn_key'] = out['inn'].map(normalize_inn_q1)
    if 'agr_id_key' not in out.columns:
        out['agr_id_key'] = out['agr_id'].map(normalize_agr_q1)
    else:
        out['agr_id_key'] = out['agr_id_key'].map(normalize_agr_q1)

    if mpos_df is None or len(mpos_df) == 0:
        out['commission_monthly'] = 0.0
        out['commission_monthly_source'] = 'mpos_rent_missing'
        return out

    m = mpos_df.copy()
    m['inn_key'] = m['inn'].map(normalize_inn_q1)
    m['agr_id_key'] = m['agr_id'].map(normalize_agr_q1)
    m['commission_monthly_mpos'] = pd.to_numeric(m['commission_monthly_mpos'], errors='coerce').fillna(0.0)
    m = (
        m.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)['commission_monthly_mpos']
        .sum()
    )

    rows_before = len(out)
    out = out.merge(m, on=['inn_key', 'agr_id_key'], how='left')
    if len(out) != rows_before:
        raise RuntimeError(f'MPOS commission join changed row count: {rows_before} -> {len(out)}')

    out['commission_monthly'] = pd.to_numeric(out['commission_monthly_mpos'], errors='coerce').fillna(0.0)
    out['commission_monthly_source'] = np.where(
        out['commission_monthly_mpos'].notna(),
        'mpos_rent_n_amt',
        'mpos_rent_zero_fill',
    )
    return out


In [ ]:
# Период расчета: Jan–Aug 2026 (июль/август — lake only, без Excel)
period_start = '2026-01-01'
period_end = '2026-08-01'

# Базовый header для Excel (для большинства месяцев)
excel_header = 0
# Точечные override по месяцам (ключ = YYYY-MM)
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
}

# Автопрогрев метаданных перед расчетом (invalidate metadata + refresh)
run_invalidate_metadata = True
run_refresh_after_invalidate = True

# False: досчитать только август с checkpoint'ов Jan–Jul (если актуальны).
# True + wipe: после смены логики amortization / active_retl_cnt.
force_recompute_final_df = False
wipe_checkpoints_on_force = False
# Проверка lake-таблицы амортизации перед месячным циклом
run_amort_source_check = True
amort_source_table = 'sandbox_ai.shestopalov_terminal_amortization_model_jan_aug'

# Ускорение 08b: используем "плоский" map c_tariff_plan -> commission_monthly_fix
# (тариф в final_df остается актуальным через секцию 09_actual_tariff_by_agr)
use_fast_flat_08b_map = True
preload_tariff_fix_flat_map = True

# Режим наполнения commission_monthly в cascade (диагностика only):
# actual (section 09) -> 08b map fallback by plan -> legacy fallback.
# Итоговый commission_monthly перезаписывается шагом 10m из MPOS_RENT.
use_legacy_commission_monthly_only = False

# Controlled batching для тяжелого шага 09_actual_tariff_by_agr.
section09_chunk_threshold = 1800
section09_chunk_size = 800

# Controlled batching для тяжелого шага 06_r2_legacy_attrs.
section06_chunk_threshold = 2200
section06_chunk_size = 900

period_months = pd.date_range(period_start, period_end, freq='MS')

excel_reference_by_month = {
    '2026-01': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx',
    '2026-02': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx',
    '2026-03': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx',
    '2026-04': '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx',
    '2026-05': '/home/jovyan/documents/Equaring/Data/05_Май_2026.xlsx',
    '2026-06': '/home/jovyan/documents/Equaring/Data/06_Июнь_2026.xlsx',
}

output_dir = Path('/home/jovyan/documents/Equaring/Data')
output_csv_path = output_dir / 'final_df_period_2026_01_2026_08_mpos.csv'
output_compare_excel_path = output_dir / 'final_df_compare_2026_01_2026_08_mpos.xlsx'
output_top10_csv_path = output_dir / 'top10_commission_delta_2026_01_2026_08_mpos.csv'
output_row_match_csv_path = output_dir / 'row_exact_match_2026_01_2026_08_mpos.csv'
output_memo_md_path = output_dir / 'memo_excel_vs_final_df_2026_01_2026_08_mpos.md'
# Month parquet/csv checkpoints: reuse Jan–Jul dir; August will be written into the same folder.
monthly_checkpoint_dir = output_dir / 'checkpoints_final_df_2026_01_2026_06_mpos'
monthly_run_state_path = monthly_checkpoint_dir / 'run_state.json'

# Kedr «Общий ЧОД» (nbi_ssp): enrich final_df_period_df before DRP
# Для обзвона август Kedr НЕ нужен — в lake достаточно 202601–202607.
run_kedr_obshiy_chod_enrich = True
kedr_obshiy_chod_table = 'sandbox_ai.shestopalov_kedr_obshiy_chod_inn_month'
kedr_chunk_size = 800
kedr_mem_limit = '8g'
kedr_overlap_csv_path = output_dir / 'kedr_obshiy_chod_src_cnt_gt1_2026_01_2026_08.csv'

# Pipeline switches for Restart & Run All
run_excel_qc = False          # TOP-10 / row-match / memo (нужны Excel-файлы)
run_drp_upload = False        # для обзвона CSV достаточно; True → upload в …_jan_jun

print('Период расчета:')
print([d.strftime('%Y-%m') for d in period_months])
print('Excel-референсы:')
for k, v in excel_reference_by_month.items():
    month_header = excel_header_by_month.get(k, excel_header)
    print(f'  {k}: {v} (header={month_header})')
print('Without Excel (lake-only):', [m for m in [d.strftime('%Y-%m') for d in period_months] if m not in excel_reference_by_month])
print(f"08b mode: {'fast_flat_map' if use_fast_flat_08b_map else 'legacy'}; preload={preload_tariff_fix_flat_map}")
print('commission_monthly mode: MPOS_RENT.n_amt (10m); cascade kept for diagnostics only')
print(f'09 batching: threshold={section09_chunk_threshold}, chunk_size={section09_chunk_size}')
print(f'checkpoint dir: {monthly_checkpoint_dir}')

# Проверка наличия Excel-файлов
for k, v in excel_reference_by_month.items():
    month_header = excel_header_by_month.get(k, excel_header)
    p = Path(v)
    print(f'Excel {k}: exists={p.exists()} | header={month_header} | {p}')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

invalidate_tables = [
    'ods_alpha.scd1_agreements', 'ods_alpha.scd1_companies', 'ods_alpha.scd1_agr_terms',
    'ocrm_ul.s_org_ext', 'cdiul.ext_id_org', 'ods_alpha.scd1_merchants',
    'ods_alpha.scd1_pos_terminals', 'sandbox_ai.shestopalov_terminal_amortization_model_jan_aug',
    'sandbox_ai.shestopalov_kedr_obshiy_chod_inn_month',
    'ods_alpha.scd1_trx', 'ods_alpha.scd1_trx_acq', 'ods_alpha.scd1_trx_int',
    'ods_alpha.scd1_base24_fiids', 'ods.scd1_z_r2_ip_merchants', 'ods.scd1_z_r2_tariff_tune',
    'ods.scd1_z_r2_tariff_fix', 'ods.scd1_z_cl_corp', 'ods.scd1_z_depart',
    'ods.scd1_z_branch', 'ods.scd1_z_r2_tariff_plan',
    'sandbox_ai.le_nbo_scoring__targets_product_agreems',
    'sandbox_ai.prod__kuznetsov_lle__acc_ul_trx_features_impala',
    'ods_alpha.scd1_mrc_pos_rent',
]

if run_invalidate_metadata:
    invalidate_failed = []
    invalidate_ok = 0

    with imp:
        for t in invalidate_tables:
            try:
                imp.execute(f'invalidate metadata {t}')
                if run_refresh_after_invalidate:
                    imp.execute(f'refresh {t}')
                invalidate_ok += 1
                print(f'[invalidate ok] {t}')
            except Exception as exc_inv:
                invalidate_failed.append((t, type(exc_inv).__name__))
                print(f'[invalidate fail] {t}: {type(exc_inv).__name__}')

    print(
        f'Invalidate metadata completed: ok={invalidate_ok}, '
        f'failed={len(invalidate_failed)}, refresh={run_refresh_after_invalidate}'
    )
    if invalidate_failed:
        print('Failed tables:')
        for t, err_type in invalidate_failed:
            print(f'  - {t}: {err_type}')
else:
    print('Invalidate metadata skipped')

# One-time flat tariff map for fast 08b (old commission logic, actual tariff still from section 09)
tariff_fix_map_flat_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])
if use_fast_flat_08b_map and preload_tariff_fix_flat_map:
    sql_tariff_fix_map_flat = """
    with tt_pairs as (
      select distinct
        tt.c_tariff_plan as c_tariff_plan_raw,
        tt.c_tariff as c_tariff_raw
      from ods.scd1_z_r2_tariff_tune tt
      where tt.c_tariff_plan is not null
    ),
    tf_agg as (
      select
        tf.id as c_tariff_raw,
        max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
      from ods.scd1_z_r2_tariff_fix tf
      group by tf.id
    )
    select
      cast(p.c_tariff_plan_raw as string) as c_tariff_plan,
      max(a.commission_monthly_fix) as commission_monthly_fix
    from tt_pairs p
    left join tf_agg a
      on p.c_tariff_raw = a.c_tariff_raw
    group by p.c_tariff_plan_raw
    """

    try:
        with imp:
            imp.execute('set MEM_LIMIT=8g')
            tariff_fix_map_flat_df = imp.fetch(sql_tariff_fix_map_flat)
    except Exception as exc_flat_map:
        print(f"08b flat map preload failed: {type(exc_flat_map).__name__}. Will fallback inside month run.")
        tariff_fix_map_flat_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])

    if tariff_fix_map_flat_df is None:
        tariff_fix_map_flat_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])

    if not tariff_fix_map_flat_df.empty:
        tariff_fix_map_flat_df['c_tariff_plan'] = tariff_fix_map_flat_df['c_tariff_plan'].astype(str).str.strip()
        tariff_fix_map_flat_df['commission_monthly_fix'] = pd.to_numeric(
            tariff_fix_map_flat_df['commission_monthly_fix'], errors='coerce'
        )
        tariff_fix_map_flat_df = (
            tariff_fix_map_flat_df
            .dropna(subset=['c_tariff_plan'])
            .groupby('c_tariff_plan', as_index=False)['commission_monthly_fix']
            .max()
            .sort_values('c_tariff_plan')
            .reset_index(drop=True)
        )

    print(
        f"08b flat map preloaded: rows={len(tariff_fix_map_flat_df):,}, "
        f"non_null_commission={int(pd.to_numeric(tariff_fix_map_flat_df.get('commission_monthly_fix'), errors='coerce').notna().sum()) if len(tariff_fix_map_flat_df) else 0:,}"
    )
else:
    print('08b flat map preload skipped')

# --- Verify amortization source in Impala (must exist before month loop) ---
if run_amort_source_check:
    sql_amort_check = f"""
    select
      count(*) as rows_cnt,
      cast(min(snapshot_month_start) as string) as min_month,
      cast(max(snapshot_month_start) as string) as max_month,
      sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as amort_sum
    from {amort_source_table}
    """
    with imp:
        amort_check_df = imp.fetch(sql_amort_check)
    print('=== Amortization source check ===')
    print('table =', amort_source_table)
    display(amort_check_df)
    if amort_check_df is None or len(amort_check_df) == 0:
        raise RuntimeError(f'Amort source check failed: empty result for {amort_source_table}')
    rows_cnt = int(pd.to_numeric(amort_check_df.iloc[0]['rows_cnt'], errors='coerce') or 0)
    if rows_cnt <= 0:
        raise RuntimeError(
            f'Amort source empty: {amort_source_table}. '
            'Run 01_07_build_amortization_drp.ipynb first.'
        )
    print(f'amort source OK: rows={rows_cnt:,}')


In [ ]:
def compute_final_df_for_month(
    report_month,
    imp,
    tariff_fix_map_flat_df=None,
    use_fast_flat_08b_map=True,
    use_legacy_commission_monthly_only=False,
    section06_chunk_threshold=2200,
    section06_chunk_size=900,
    section09_chunk_threshold=1800,
    section09_chunk_size=800,
):
    report_month_ts = pd.to_datetime(report_month)
    month_start = report_month_ts.strftime('%Y-%m-%d')
    month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
    report_month_label = report_month_ts.strftime('%Y-%m')
    snapshot_month_start = month_start

    month_total_start_ts = time.perf_counter()

    def _log_progress(message):
        print(f"[{report_month_label}][{time.strftime('%H:%M:%S')}] {message}", flush=True)

    # Быстрый диагноз "очередь vs план" за ~1-2 минуты:
    # - быстрый probe select 1 перед тяжелым SQL;
    # - если probe медленный -> вероятнее waiting/queue pressure;
    # - если probe быстрый, а шаг долгий -> вероятнее тяжелый план/скан данных.
    enable_waiting_diagnose = True
    diagnose_queue_warn_sec = 15
    diagnose_long_step_sec = 120

    def _quick_queue_probe(mem_limit='8g'):
        probe_start_ts = time.perf_counter()
        with imp:
            imp.execute(f'set MEM_LIMIT={mem_limit}')
            imp.fetch('select 1 as probe_ping')
        return round(time.perf_counter() - probe_start_ts, 2)

    def _run_impala_fetch(step_name, sql_text, mem_limit='8g', heartbeat_sec=60, run_probe=True):
        step_start_ts = time.perf_counter()
        _log_progress(f"{step_name}: start")

        queue_probe_sec = None
        if enable_waiting_diagnose and run_probe:
            try:
                queue_probe_sec = _quick_queue_probe(mem_limit=mem_limit)
                if queue_probe_sec >= diagnose_queue_warn_sec:
                    _log_progress(
                        f"{step_name}: queue probe={queue_probe_sec}s -> вероятно waiting/очередь"
                    )
                else:
                    _log_progress(
                        f"{step_name}: queue probe={queue_probe_sec}s -> кластер отвечает быстро"
                    )
            except Exception as probe_exc:
                _log_progress(
                    f"{step_name}: queue probe failed ({type(probe_exc).__name__}), продолжаю основной запрос"
                )

        heartbeat_stop = threading.Event()

        def _heartbeat():
            while not heartbeat_stop.wait(heartbeat_sec):
                running_sec = round(time.perf_counter() - step_start_ts, 1)
                if queue_probe_sec is not None and running_sec >= diagnose_long_step_sec:
                    if queue_probe_sec >= diagnose_queue_warn_sec:
                        _log_progress(f"{step_name}: still running ({running_sec}s) [hint: waiting/queue likely]")
                    else:
                        _log_progress(f"{step_name}: still running ({running_sec}s) [hint: slow plan/scan likely]")
                else:
                    _log_progress(f"{step_name}: still running ({running_sec}s)")

        heartbeat_thread = threading.Thread(target=_heartbeat, daemon=True)
        heartbeat_thread.start()

        try:
            with imp:
                imp.execute(f'set MEM_LIMIT={mem_limit}')
                df_out = imp.fetch(sql_text)
        except Exception:
            elapsed_sec = round(time.perf_counter() - step_start_ts, 2)
            _log_progress(f"{step_name}: failed after {elapsed_sec}s")
            raise
        finally:
            heartbeat_stop.set()
            heartbeat_thread.join(timeout=0.2)

        elapsed_sec = round(time.perf_counter() - step_start_ts, 2)
        rows = len(df_out) if isinstance(df_out, pd.DataFrame) else 0
        _log_progress(f"{step_name}: done ({elapsed_sec}s, rows={rows:,})")

        if queue_probe_sec is not None and elapsed_sec >= diagnose_long_step_sec:
            if queue_probe_sec >= diagnose_queue_warn_sec:
                _log_progress(
                    f"{step_name}: quick-diagnose => скорее waiting/очередь (probe={queue_probe_sec}s)"
                )
            else:
                _log_progress(
                    f"{step_name}: quick-diagnose => скорее медленный план/большой скан (probe={queue_probe_sec}s)"
                )

        return df_out

    section_elapsed_tracker = {}

    def _save_section_elapsed(section_key, start_ts):
        elapsed = round(time.perf_counter() - start_ts, 2)
        section_elapsed_tracker[section_key] = elapsed
        _log_progress(f"section_{section_key}_elapsed_sec = {elapsed}")
        return elapsed

    def _split_scope(values, chunk_size):
        scope_values = [str(x).strip() for x in (values or []) if str(x).strip()]
        if not scope_values:
            return []
        safe_chunk_size = max(int(chunk_size or 1), 1)
        return [scope_values[i:i + safe_chunk_size] for i in range(0, len(scope_values), safe_chunk_size)]

    def _in_sql_list(values):
        values = [str(x) for x in (values or []) if str(x).strip()]
        return ', '.join([f"'{x}'" for x in values]) if values else "''"

    _log_progress('start расчет final_df')

    # 01_sa_perimeter
    section01_start_ts = time.perf_counter()
    sql_sa_perimeter = f"""
    select distinct
      cast(a.n_agr as string) as n_agr,
      cast(a.abs_agr_id as string) as agr_id,
      cast(a.n_cmp_client as string) as n_cmp_client,
      cast(a.c_agr_number as string) as contract_number,
      cast(a.d_valid_from as date) as d_valid_from,
      cast(a.d_valid_to as date) as d_valid_to,
      regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
      cast(c.c_cmp_name as string) as company_name
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    where upper(trim(cast(a.acq_class as string))) = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
      and c.c_inn is not null
      and exists (
          select 1
          from ods_alpha.scd1_agr_terms t
          where cast(t.n_agr as string) = cast(a.n_agr as string)
            and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
            and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
            and upper(trim(cast(t.cf_ter_type as string))) = 'P'
            and coalesce(t.ods_deleted_flg, '0') <> '1'
      )
    """

    sa_df = _run_impala_fetch('01_sa_perimeter', sql_sa_perimeter, mem_limit='8g')

    if sa_df is None:
        sa_df = pd.DataFrame()
    if not sa_df.empty:
        sa_df['inn'] = sa_df['inn'].map(normalize_inn)
        sa_df['contract_number'] = sa_df['contract_number'].map(normalize_contract)

    section01_elapsed_sec = _save_section_elapsed('01_sa_perimeter', section01_start_ts)

    # 02_cdi_map
    section02_start_ts = time.perf_counter()
    inn_values = sorted([
        x for x in sa_df.get('inn', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x
    ])
    def _build_sql_cdi(inn_scope):
        inn_sql_list = _in_sql_list(inn_scope)
        return f"""
        with ocrm_current as (
          select
            regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
            cast(soe.row_id as string) as row_id,
            trim(cast(soe.x_area_resp as string)) as x_area_resp_norm,
            trim(cast(soe.x_area_resp as string)) as ssp_ocrm,
            row_number() over (
              partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
              order by cast(soe.created as timestamp) desc, cast(soe.row_id as string) desc
            ) as rn
          from ocrm_ul.s_org_ext soe
          where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_sql_list})
            and coalesce(soe.x_removed_flg, 'N') = 'N'
            and coalesce(soe.x_duplicate_flg, 'N') = 'N'
        ),
        ocrm_one as (
          select inn, row_id, ssp_ocrm, x_area_resp_norm
          from ocrm_current
          where rn = 1
        )
        select
          o.inn,
          o.ssp_ocrm,
          o.x_area_resp_norm,
          cast(e.party_id as string) as cdi_id
        from ocrm_one o
        left join cdiul.ext_id_org e
          on cast(e.cmo_ext_party_source_id as string) = o.row_id
         and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
        """

    if not inn_values:
        cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm_raw', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])
    else:
        cdi_map_df = _run_impala_fetch('02_cdi_map', _build_sql_cdi(inn_values), mem_limit='8g')

    if cdi_map_df is None:
        cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm_raw', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])
    if not cdi_map_df.empty:
        cdi_map_df['inn'] = cdi_map_df['inn'].map(normalize_inn)
        cdi_map_df['cdi_id'] = cdi_map_df['cdi_id'].astype(str)
        cdi_map_df['x_area_resp_norm'] = cdi_map_df['x_area_resp_norm'].apply(normalize_text_value)
        cdi_map_df['ssp_ocrm_raw'] = cdi_map_df['x_area_resp_norm']
        cdi_map_df['ssp_ocrm'] = cdi_map_df['ssp_ocrm_raw'].apply(normalize_ssp_ocrm_core)
        cdi_map_df = cdi_map_df.drop_duplicates(subset=['inn'], keep='first')

    section02_elapsed_sec = _save_section_elapsed('02_cdi_map', section02_start_ts)

    # 03_cft_map
    section03_start_ts = time.perf_counter()
    cdi_values = sorted([
        x for x in cdi_map_df.get('cdi_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x
    ])
    def _build_sql_cft(cdi_scope):
        cdi_sql_list = _in_sql_list(cdi_scope)
        return f"""
        select
          cast(e.party_id as string) as cdi_id,
          cast(e.cmo_ext_party_source_id as string) as cft_id
        from cdiul.ext_id_org e
        where cast(e.party_id as string) in ({cdi_sql_list})
          and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
        """

    if not cdi_values:
        cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])
    else:
        cft_map_df = _run_impala_fetch('03_cft_map', _build_sql_cft(cdi_values), mem_limit='8g')

    if cft_map_df is None:
        cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])
    if not cft_map_df.empty:
        cft_map_df['cdi_id'] = cft_map_df['cdi_id'].astype(str)
        cft_map_df['cft_id'] = cft_map_df['cft_id'].astype(str)
        cft_map_df = cft_map_df.drop_duplicates(subset=['cdi_id'], keep='first')

    section03_elapsed_sec = _save_section_elapsed('03_cft_map', section03_start_ts)

    # 04_operational_metrics (hybrid_all optimized: 547-like core + fallback only for missing n_agr)
    section04_start_ts = time.perf_counter()
    if sa_df.empty:
        cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization', 'term_calc_source'])
    else:
        def _build_sql_cmp_hybrid(use_m_acq_filter=True):
            m_acq_filter_sql = "and coalesce(upper(trim(cast(m.acq_class as string))), 'NA') <> 'SV'" if use_m_acq_filter else ''

            return f"""
            with sa_agr as (
              select distinct
                cast(a.n_agr as string) as n_agr,
                cast(a.n_cmp_client as string) as n_cmp_client
              from ods_alpha.scd1_agreements a
              join ods_alpha.scd1_companies c
                on c.n_cmp = a.n_cmp_client
              where upper(trim(cast(a.acq_class as string))) = 'SA'
                and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
                and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
                and coalesce(a.ods_deleted_flg, '0') <> '1'
                and coalesce(c.ods_deleted_flg, '0') <> '1'
                and c.c_inn is not null
                and exists (
                  select 1
                  from ods_alpha.scd1_agr_terms t
                  where cast(t.n_agr as string) = cast(a.n_agr as string)
                    and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
                    and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
                    and upper(trim(cast(t.cf_ter_type as string))) = 'P'
                    and coalesce(t.ods_deleted_flg, '0') <> '1'
                )
            ),

            terms_547 as (
              select distinct
                sa.n_agr,
                sa.n_cmp_client,
                cast(t.c_nmrc as string) as c_nmrc
              from sa_agr sa
              join ods_alpha.scd1_agr_terms t
                on cast(t.n_agr as string) = sa.n_agr
              join ods_alpha.scd1_merchants m
                on cast(m.c_nmrc as string) = cast(t.c_nmrc as string)
              where t.c_nmrc is not null
                and upper(coalesce(trim(cast(m.c_mrc_name as string)), '')) not like 'REZERVNYI TERMINAL%'
                {m_acq_filter_sql}
                and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
                and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
                and coalesce(t.ods_deleted_flg, '0') <> '1'
                and coalesce(m.ods_deleted_flg, '0') <> '1'
            ),
            pos_period as (
              select distinct
                cast(p.c_nmrc as string) as c_nmrc,
                cast(p.c_pos_serial as string) as c_pos_serial,
                cast(p.c_nter as string) as c_nter
              from ods_alpha.scd1_pos_terminals p
              where p.c_pos_serial is not null
                and cast(p.d_ter_install as date) is not null
                and cast(p.d_ter_install as date) < date_add(cast('{month_end}' as date), 1)
                and (p.d_ter_close is null or cast(p.d_ter_close as date) >= cast('{month_start}' as date))
                and coalesce(p.ods_deleted_flg, '0') <> '1'
            ),
            term_active_547 as (
              select
                t.n_agr,
                t.n_cmp_client,
                t.c_nmrc,
                p.c_pos_serial,
                p.c_nter
              from terms_547 t
              left join pos_period p
                on p.c_nmrc = t.c_nmrc
            ),
            retl_547 as (
              select n_agr, count(distinct c_nmrc) as retl_cnt_547
              from term_active_547
              group by n_agr
            ),
            term_547 as (
              select n_agr, count(distinct c_pos_serial) as term_cnt_547
              from term_active_547
              group by n_agr
            ),
            -- One physical device (serial) -> one owner n_agr -> one amort amount.
            -- Avoids multi-agr and mid-month c_nter rename double-count.
            amort_serial_owner_547 as (
              select n_agr, c_nter, c_pos_serial
              from (
                select
                  cast(n_agr as string) as n_agr,
                  cast(c_nter as string) as c_nter,
                  cast(c_pos_serial as string) as c_pos_serial,
                  row_number() over (
                    partition by coalesce(
                      nullif(trim(cast(c_pos_serial as string)), ''),
                      concat('__NTER__', cast(c_nter as string))
                    )
                    order by
                      cast(n_agr as string),
                      cast(c_nter as string)
                  ) as rn
                from term_active_547
                where c_nter is not null
              ) z
              where rn = 1
            ),
            amort_547 as (
              select
                o.n_agr,
                sum(coalesce(cast(am.amortization_for_report_month as double), 0.0)) as amortization_547
              from amort_serial_owner_547 o
              left join {amort_source_table} am
                on cast(am.c_nter as string) = o.c_nter
               and cast(am.snapshot_month_start as date) = cast('{month_start}' as date)
              group by o.n_agr
            ),

            fallback_agrs as (
              select sa.n_agr, sa.n_cmp_client
              from sa_agr sa
              left join retl_547 r547 on r547.n_agr = sa.n_agr
              left join term_547 t547 on t547.n_agr = sa.n_agr
              where r547.n_agr is null and t547.n_agr is null
            ),
            old_nmrc as (
              select fa.n_agr, fa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
              from fallback_agrs fa
              join ods_alpha.scd1_merchants mm
                on cast(mm.n_cmp as string) = fa.n_cmp_client
              where mm.c_nmrc is not null
                and coalesce(mm.ods_deleted_flg, '0') <> '1'
              group by fa.n_agr, fa.n_cmp_client, cast(mm.c_nmrc as string)
            ),
            old_term_active as (
              select
                o.n_agr,
                o.n_cmp_client,
                o.c_nmrc,
                cast(t.c_nter as string) as c_nter,
                cast(t.c_pos_serial as string) as c_pos_serial
              from old_nmrc o
              join ods_alpha.scd1_pos_terminals t
                on cast(t.c_nmrc as string) = o.c_nmrc
              where t.c_nter is not null
                and coalesce(t.ods_deleted_flg, '0') <> '1'
                and cast(t.d_ter_install as date) is not null
                and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
                and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
              group by o.n_agr, o.n_cmp_client, o.c_nmrc, cast(t.c_nter as string), cast(t.c_pos_serial as string)
            ),
            old_retl as (
              select n_agr, count(distinct c_nmrc) as retl_cnt_old
              from old_term_active
              group by n_agr
            ),
            old_term as (
              select n_agr, count(distinct c_nter) as term_cnt_old
              from old_term_active
              group by n_agr
            ),
            amort_serial_owner_old as (
              select n_agr, c_nter, c_pos_serial
              from (
                select
                  cast(n_agr as string) as n_agr,
                  cast(c_nter as string) as c_nter,
                  cast(c_pos_serial as string) as c_pos_serial,
                  row_number() over (
                    partition by coalesce(
                      nullif(trim(cast(c_pos_serial as string)), ''),
                      concat('__NTER__', cast(c_nter as string))
                    )
                    order by
                      cast(n_agr as string),
                      cast(c_nter as string)
                  ) as rn
                from old_term_active
                where c_nter is not null
              ) z
              where rn = 1
            ),
            old_amort as (
              select
                o.n_agr,
                sum(coalesce(cast(am.amortization_for_report_month as double), 0.0)) as amortization_old
              from amort_serial_owner_old o
              left join {amort_source_table} am
                on cast(am.c_nter as string) = o.c_nter
               and cast(am.snapshot_month_start as date) = cast('{month_start}' as date)
              group by o.n_agr
            )

            select
              sa.n_agr,
              sa.n_cmp_client,
              coalesce(r547.retl_cnt_547, rold.retl_cnt_old) as retl_cnt,
              coalesce(t547.term_cnt_547, told.term_cnt_old) as term_cnt,
              coalesce(a547.amortization_547, aold.amortization_old, 0.0) as amortization,
              case
                when r547.n_agr is not null or t547.n_agr is not null then '547_like'
                when rold.n_agr is not null or told.n_agr is not null then 'fallback_old'
                else 'no_data'
              end as term_calc_source
            from sa_agr sa
            left join retl_547 r547 on r547.n_agr = sa.n_agr
            left join term_547 t547 on t547.n_agr = sa.n_agr
            left join amort_547 a547 on a547.n_agr = sa.n_agr
            left join old_retl rold on rold.n_agr = sa.n_agr
            left join old_term told on told.n_agr = sa.n_agr
            left join old_amort aold on aold.n_agr = sa.n_agr
            """

        used_m_acq_filter = True
        try:
            cmp_df = _run_impala_fetch('04_operational_metrics_hybrid', _build_sql_cmp_hybrid(use_m_acq_filter=True), mem_limit='16g')
        except Exception as exc_hybrid:
            used_m_acq_filter = False
            _log_progress(f"04_operational_metrics_hybrid: m.acq_class filter disabled due to {type(exc_hybrid).__name__}")
            cmp_df = _run_impala_fetch('04_operational_metrics_hybrid_fallback', _build_sql_cmp_hybrid(use_m_acq_filter=False), mem_limit='16g')

    if cmp_df is None:
        cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization', 'term_calc_source'])
    if not cmp_df.empty:
        cmp_df['n_agr'] = cmp_df['n_agr'].astype(str)
        cmp_df['n_cmp_client'] = cmp_df['n_cmp_client'].astype(str)
        cmp_df['retl_cnt'] = pd.to_numeric(cmp_df['retl_cnt'], errors='coerce')
        cmp_df['term_cnt'] = pd.to_numeric(cmp_df['term_cnt'], errors='coerce')
        cmp_df['amortization'] = pd.to_numeric(cmp_df['amortization'], errors='coerce')

    section04_elapsed_sec = _save_section_elapsed('04_operational_metrics', section04_start_ts)

    # 05_transaction_metrics
    section05_start_ts = time.perf_counter()
    if sa_df.empty:
        trx_df = pd.DataFrame(columns=['n_agr', 'inn', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt', 'active_terms'])
    else:
        sql_trx = f"""
        with sa_agr as (
          select distinct
            cast(a.n_agr as string) as n_agr,
            cast(a.n_cmp_client as string) as n_cmp_client,
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
          from ods_alpha.scd1_agreements a
          join ods_alpha.scd1_companies c
            on c.n_cmp = a.n_cmp_client
          where upper(trim(cast(a.acq_class as string))) = 'SA'
            and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
            and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
            and coalesce(a.ods_deleted_flg, '0') <> '1'
            and coalesce(c.ods_deleted_flg, '0') <> '1'
            and c.c_inn is not null
            and exists (
              select 1
              from ods_alpha.scd1_agr_terms t
              where cast(t.n_agr as string) = cast(a.n_agr as string)
                and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
                and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
                and upper(trim(cast(t.cf_ter_type as string))) = 'P'
                and coalesce(t.ods_deleted_flg, '0') <> '1'
            )
        ),
        fiid_rshb as (
          select distinct cast(fa.c_fiid as string) as c_fiid
          from ods_alpha.scd1_base24_fiids fa
          where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
        ),
        trx_base_raw as (
          select cast(t.n_trx as string) as n_trx, cast(t.c_nter as string) as c_nter, cast(t.n_amt_src as double) as n_amt_src
          from ods_alpha.scd1_trx t
          join fiid_rshb fr
            on fr.c_fiid = cast(t.c_fiid_acq as string)
          where cast(t.d_trx_orig as timestamp) >= cast('{month_start}' as timestamp)
            and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{month_end}' as date), 1) as timestamp)
            and t.c_nter is not null
            and coalesce(t.ods_deleted_flg, '0') <> '1'
            and t.c_trx_class = 'SA'
            and t.c_trx_type = 'S01'
            and coalesce(t.cf_trx_stat, '') <> 'R'
        ),
        trx_base as (
          select n_trx, max(c_nter) as c_nter, max(n_amt_src) as n_amt_src
          from trx_base_raw
          group by n_trx
        ),
        ta_raw as (
          select cast(a.n_trx as string) as n_trx, cast(a.n_agr as string) as n_agr, coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
          from ods_alpha.scd1_trx_acq a
          join trx_base tb on tb.n_trx = cast(a.n_trx as string)
          join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
        ),
        ta as (
          select n_trx, n_agr, max(n_amt_tax) as n_amt_tax
          from ta_raw
          group by n_trx, n_agr
        ),
        trx_keys as (
          select distinct n_trx
          from ta
        ),
        trx_int_agg as (
          -- hotfix March 2026: scd1_trx_int had exact duplicate alive rows per n_trx;
          -- sum(n_amt_fee) doubled IRF (~-6M). max == dedupe same-fee (100% of multi-int).
          select cast(i.n_trx as string) as n_trx, max(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
          from ods_alpha.scd1_trx_int i
          join trx_keys k on k.n_trx = cast(i.n_trx as string)
          where coalesce(i.ods_deleted_flg, '0') <> '1'
          group by cast(i.n_trx as string)
        ),
        tj as (
          select ta.n_agr, sa.n_cmp_client, sa.inn, ta.n_trx, tb.c_nter, tb.n_amt_src, ta.n_amt_tax
          from ta
          join trx_base tb on tb.n_trx = ta.n_trx
          left join sa_agr sa on sa.n_agr = ta.n_agr
        ),
        trx_agg as (
          select
            tj.n_agr,
            count(distinct tj.n_trx) as trx_cnt,
            sum(tj.n_amt_src) as trx_sum,
            sum(tj.n_amt_tax) as commission_from_ops,
            sum(coalesce(i.n_amt_fee, 0.0)) as int_component
          from tj
          left join trx_int_agg i on i.n_trx = tj.n_trx
          group by tj.n_agr
        ),
        active_term_agg_ncmp as (
          select
            tj.n_cmp_client,
            count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_term_cnt
          from tj
          where tj.n_cmp_client is not null
          group by tj.n_cmp_client
        ),
        active_term_agg_agr as (
          select
            tj.n_agr,
            count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_terms
          from tj
          where tj.n_agr is not null
          group by tj.n_agr
        )
        select
          t.n_agr,
          sa.inn,
          t.trx_cnt,
          t.trx_sum,
          t.commission_from_ops,
          t.int_component,
          sa.n_cmp_client,
          a.active_term_cnt,
          aa.active_terms
        from trx_agg t
        left join sa_agr sa on sa.n_agr = t.n_agr
        left join active_term_agg_ncmp a on a.n_cmp_client = sa.n_cmp_client
        left join active_term_agg_agr aa on aa.n_agr = t.n_agr
        """

        trx_df = _run_impala_fetch('05_transaction_metrics', sql_trx, mem_limit='16g')

    if trx_df is None:
        trx_df = pd.DataFrame(columns=['n_agr', 'inn', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt', 'active_terms'])
    if not trx_df.empty:
        trx_df['n_agr'] = trx_df['n_agr'].astype(str)
        trx_df['n_cmp_client'] = trx_df['n_cmp_client'].astype(str)
        trx_df['inn'] = trx_df['inn'].map(normalize_inn)
        trx_df['active_term_cnt'] = pd.to_numeric(trx_df['active_term_cnt'], errors='coerce')
        trx_df['active_terms'] = pd.to_numeric(trx_df['active_terms'], errors='coerce')

    active_term_df = (
        trx_df[['n_cmp_client', 'active_term_cnt']]
        .dropna(subset=['n_cmp_client'])
        .drop_duplicates(subset=['n_cmp_client'])
        if len(trx_df)
        else pd.DataFrame(columns=['n_cmp_client', 'active_term_cnt'])
    )

    section05_elapsed_sec = _save_section_elapsed('05_transaction_metrics', section05_start_ts)

    # 06_r2_legacy_attrs
    section06_start_ts = time.perf_counter()
    cft_values = sorted([
        x for x in cft_map_df.get('cft_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x
    ])
    if not cft_values:
        r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'c_tariff_plan', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf_raw', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])
    else:
        # Scoped R2 lookup by current cft perimeter with early dedup per cft_id.
        # В optimized-режиме стараемся избежать CAST на левой стороне большого скана.
        def _split_cft_scope(cft_scope):
            numeric_scope = []
            text_scope = []
            for raw_v in (cft_scope or []):
                v = str(raw_v).strip()
                if not v:
                    continue
                if v.isdigit():
                    numeric_scope.append(v)
                else:
                    text_scope.append(v)
            return sorted(set(numeric_scope)), sorted(set(text_scope))

        def _build_sql_r2_scoped(cft_scope):
            cft_numeric_scope, cft_text_scope = _split_cft_scope(cft_scope)

            scope_predicates = []
            if cft_numeric_scope:
                scope_predicates.append(f"m.c_cl_org in ({', '.join(cft_numeric_scope)})")
            if cft_text_scope:
                cft_text_sql_list = _in_sql_list(cft_text_scope)
                scope_predicates.append(f"cast(m.c_cl_org as string) in ({cft_text_sql_list})")

            scope_predicate_sql = ' or '.join([f"({p})" for p in scope_predicates]) if scope_predicates else '1 = 0'

            return f"""
            with merchant_ranked as (
              select
                m.id as r2_id_raw,
                m.c_cl_org as cft_id_raw,
                m.c_depart as c_depart_raw,
                m.c_tariff_plan as c_tariff_plan_raw,
                row_number() over (
                  partition by m.c_cl_org
                  order by
                    case when m.c_tariff_plan is not null then 0 else 1 end,
                    case when m.c_depart is not null then 0 else 1 end,
                    m.id desc
                ) as rn
              from ods.scd1_z_r2_ip_merchants m
              where ({scope_predicate_sql})
                and m.c_cl_org is not null
                and coalesce(m.ods_deleted_flg, '0') <> '1'
            ),
            r2_raw as (
              select
                r2_id_raw,
                cft_id_raw,
                c_depart_raw,
                c_tariff_plan_raw
              from merchant_ranked
              where rn = 1
            ),
            r2_enriched as (
              select
                r2.r2_id_raw as r2_id,
                r2.cft_id_raw as cft_id,
                r2.c_tariff_plan_raw as c_tariff_plan,
                corp.c_register_gos_reg_num_rec as ogrn,
                dep.c_name as vsp_name,
                dep.c_num as vsp_code,
                br.c_shortlabel as filial_rf_raw,
                tp.c_name as tariff_name_legacy,
                cast(null as decimal(18,2)) as commission_monthly_legacy
              from r2_raw r2
              left join ods.scd1_z_cl_corp corp on corp.id = r2.cft_id_raw
              left join ods.scd1_z_depart dep on dep.id = r2.c_depart_raw
              left join ods.scd1_z_branch br on br.id = dep.c_filial
              left join ods.scd1_z_r2_tariff_plan tp on tp.id = r2.c_tariff_plan_raw
            )
            select
              r2_id,
              cft_id,
              c_tariff_plan,
              ogrn,
              vsp_name,
              vsp_code,
              filial_rf_raw,
              tariff_name_legacy,
              commission_monthly_legacy
            from r2_enriched
            """

        def _build_sql_r2_full(cft_scope):
            # Fallback keeps legacy CAST-based matching if native-type joins fail.
            cft_sql_list = _in_sql_list(cft_scope)
            return f"""
            with merchant_ranked as (
              select
                m.id as r2_id_raw,
                m.c_cl_org as cft_id_raw,
                m.c_depart as c_depart_raw,
                m.c_tariff_plan as c_tariff_plan_raw,
                row_number() over (
                  partition by cast(m.c_cl_org as string)
                  order by
                    case when m.c_tariff_plan is not null then 0 else 1 end,
                    case when m.c_depart is not null then 0 else 1 end,
                    m.id desc
                ) as rn
              from ods.scd1_z_r2_ip_merchants m
              where cast(m.c_cl_org as string) in ({cft_sql_list})
                and m.c_cl_org is not null
                and coalesce(m.ods_deleted_flg, '0') <> '1'
            ),
            r2_raw as (
              select
                r2_id_raw,
                cft_id_raw,
                c_depart_raw,
                c_tariff_plan_raw
              from merchant_ranked
              where rn = 1
            ),
            r2_enriched as (
              select
                r2.r2_id_raw as r2_id,
                r2.cft_id_raw as cft_id,
                r2.c_tariff_plan_raw as c_tariff_plan,
                corp.c_register_gos_reg_num_rec as ogrn,
                dep.c_name as vsp_name,
                dep.c_num as vsp_code,
                br.c_shortlabel as filial_rf_raw,
                tp.c_name as tariff_name_legacy,
                cast(null as decimal(18,2)) as commission_monthly_legacy
              from r2_raw r2
              left join ods.scd1_z_cl_corp corp on corp.id = r2.cft_id_raw
              left join ods.scd1_z_depart dep on dep.id = r2.c_depart_raw
              left join ods.scd1_z_branch br on br.id = dep.c_filial
              left join ods.scd1_z_r2_tariff_plan tp on tp.id = r2.c_tariff_plan_raw
            )
            select
              r2_id,
              cft_id,
              c_tariff_plan,
              ogrn,
              vsp_name,
              vsp_code,
              filial_rf_raw,
              tariff_name_legacy,
              commission_monthly_legacy
            from r2_enriched
            """

        def _run_r2_scope_fetch(step_prefix, cft_scope, run_probe=True):
            try:
                return _run_impala_fetch(
                    f"{step_prefix}_scoped",
                    _build_sql_r2_scoped(cft_scope),
                    mem_limit='8g',
                    run_probe=run_probe,
                )
            except Exception as exc_r2_scoped:
                _log_progress(f"{step_prefix}_scoped: failed ({type(exc_r2_scoped).__name__}); fallback to full scoped copy")
                return _run_impala_fetch(
                    f"{step_prefix}_full_fallback",
                    _build_sql_r2_full(cft_scope),
                    mem_limit='8g',
                    run_probe=run_probe,
                )

        _log_progress(
            f"06_r2_legacy_attrs: single-shot fetch for cft scope={len(cft_values):,}"
        )
        r2_df = _run_r2_scope_fetch('06_r2_legacy_attrs_single', cft_values, run_probe=True)

        section06_scope_debug = {
            'report_month': report_month_label,
            'mode': 'single_shot',
            'cft_scope_count': int(len(cft_values)),
        }
        globals().setdefault('section06_scope_debug_by_month', {})
        globals()['section06_scope_debug_by_month'][report_month_label] = section06_scope_debug

    if r2_df is None:
        r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'c_tariff_plan', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf_raw', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])

    if not r2_df.empty:
        for c in ['r2_id', 'cft_id', 'c_tariff_plan', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf_raw', 'tariff_name_legacy']:
            r2_df[c] = r2_df[c].astype(str)

        r2_df['filial_rf_raw'] = r2_df['filial_rf_raw'].apply(normalize_text_value)
        r2_df['filial_rf'] = r2_df['filial_rf_raw'].apply(normalize_filial_rf)

        if 'commission_monthly_legacy' not in r2_df.columns:
            r2_df['commission_monthly_legacy'] = None
        r2_df['commission_monthly_legacy'] = pd.to_numeric(r2_df.get('commission_monthly_legacy'), errors='coerce')

        # Комиссию считаем позже в секции 09 по актуальному списку agr_id -> c_tariff_plan.
        _log_progress('06_r2_legacy_commission_map: skipped (moved to section 09 actual plan scope)')

        r2_profile_cft_df = (
            r2_df.groupby('cft_id', as_index=False)
            .agg(
                filial_rf_cft_fb=('filial_rf', first_non_empty_value),
                vsp_name_cft_fb=('vsp_name', first_non_empty_value),
                vsp_code_cft_fb=('vsp_code', first_non_empty_value),
            )
        )
        r2_profile_r2_df = (
            r2_df.groupby('r2_id', as_index=False)
            .agg(
                filial_rf_r2_fb=('filial_rf', first_non_empty_value),
                vsp_name_r2_fb=('vsp_name', first_non_empty_value),
                vsp_code_r2_fb=('vsp_code', first_non_empty_value),
            )
        )

        r2_df = r2_df.drop_duplicates(subset=['cft_id'], keep='first')

        comm_fill_pct = round(100.0 * pd.to_numeric(r2_df.get('commission_monthly_legacy'), errors='coerce').notna().mean(), 2)
        tariff_fill_pct = round(100.0 * (r2_df.get('tariff_name_legacy').notna() & (r2_df.get('tariff_name_legacy').astype(str).str.strip() != '')).mean(), 2)

        _log_progress(
            f"06_r2_legacy_attrs: cft_scope={len(cft_values):,}, rows_after_dedup={len(r2_df):,}, tariff_fill={tariff_fill_pct}%, commission_monthly_fill={comm_fill_pct}%"
        )
    else:
        r2_profile_cft_df = pd.DataFrame(columns=['cft_id', 'filial_rf_cft_fb', 'vsp_name_cft_fb', 'vsp_code_cft_fb'])
        r2_profile_r2_df = pd.DataFrame(columns=['r2_id', 'filial_rf_r2_fb', 'vsp_name_r2_fb', 'vsp_code_r2_fb'])

    section06_elapsed_sec = _save_section_elapsed('06_r2_legacy_attrs', section06_start_ts)

    # 07_base_merge
    section07_start_ts = time.perf_counter()
    def _is_blank_series(s):
        return s.isna() | s.astype(str).str.strip().isin(['', 'nan', 'None'])

    base_df = sa_df.copy()

    if not cdi_map_df.empty and not base_df.empty:
        cdi_merge_cols = ['inn', 'cdi_id', 'ssp_ocrm']
        for extra_col in ['ssp_ocrm_raw', 'x_area_resp_norm']:
            if extra_col in cdi_map_df.columns:
                cdi_merge_cols.append(extra_col)
        base_df = base_df.merge(cdi_map_df[cdi_merge_cols], on='inn', how='left')
    else:
        base_df['cdi_id'] = None
        base_df['ssp_ocrm'] = None
        base_df['ssp_ocrm_raw'] = None
        base_df['x_area_resp_norm'] = None

    if 'ssp_ocrm_raw' in base_df.columns:
        ssp_from_raw = base_df['ssp_ocrm_raw'].apply(normalize_ssp_ocrm_core)
        base_df['ssp_ocrm'] = ssp_from_raw.where(ssp_from_raw.notna(), base_df['ssp_ocrm'])
    base_df['ssp_ocrm'] = base_df['ssp_ocrm'].apply(normalize_ssp_ocrm_core)

    if not cft_map_df.empty and not base_df.empty:
        base_df = base_df.merge(cft_map_df[['cdi_id', 'cft_id']], on='cdi_id', how='left')
    else:
        base_df['cft_id'] = None

    if not r2_df.empty and not base_df.empty:
        base_df = base_df.merge(r2_df, on='cft_id', how='left')
    else:
        for col in ['r2_id', 'c_tariff_plan', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf_raw', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy']:
            base_df[col] = None

    if not base_df.empty and 'cft_id' in base_df.columns and 'r2_profile_cft_df' in locals() and not r2_profile_cft_df.empty:
        base_df = base_df.merge(r2_profile_cft_df, on='cft_id', how='left')
    else:
        for col in ['filial_rf_cft_fb', 'vsp_name_cft_fb', 'vsp_code_cft_fb']:
            base_df[col] = None

    if not base_df.empty and 'r2_id' in base_df.columns and 'r2_profile_r2_df' in locals() and not r2_profile_r2_df.empty:
        base_df = base_df.merge(r2_profile_r2_df, on='r2_id', how='left')
    else:
        for col in ['filial_rf_r2_fb', 'vsp_name_r2_fb', 'vsp_code_r2_fb']:
            base_df[col] = None

    for col, cft_fb, r2_fb in [
        ('filial_rf', 'filial_rf_cft_fb', 'filial_rf_r2_fb'),
        ('vsp_name', 'vsp_name_cft_fb', 'vsp_name_r2_fb'),
        ('vsp_code', 'vsp_code_cft_fb', 'vsp_code_r2_fb'),
    ]:
        missing_mask = _is_blank_series(base_df[col])
        base_df.loc[missing_mask, col] = base_df.loc[missing_mask, cft_fb]

        missing_mask = _is_blank_series(base_df[col])
        base_df.loc[missing_mask, col] = base_df.loc[missing_mask, r2_fb]

    base_df['filial_rf_raw'] = base_df['filial_rf_raw'].apply(normalize_text_value)
    base_df['filial_rf'] = base_df['filial_rf'].apply(normalize_filial_rf)

    for fb_col in ['filial_rf_cft_fb', 'vsp_name_cft_fb', 'vsp_code_cft_fb', 'filial_rf_r2_fb', 'vsp_name_r2_fb', 'vsp_code_r2_fb']:
        if fb_col in base_df.columns:
            base_df = base_df.drop(columns=[fb_col])

    if not cmp_df.empty and not base_df.empty:
        base_df = base_df.merge(cmp_df[['n_agr', 'retl_cnt', 'term_cnt', 'amortization']], on='n_agr', how='left')
    else:
        for col in ['retl_cnt', 'term_cnt', 'amortization']:
            base_df[col] = None

    if not active_term_df.empty and not base_df.empty:
        base_df = base_df.merge(active_term_df[['n_cmp_client', 'active_term_cnt']], on='n_cmp_client', how='left')
    else:
        base_df['active_term_cnt'] = None

    if not trx_df.empty and not base_df.empty:
        base_df = base_df.merge(
            trx_df[['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'active_terms']],
            on='n_agr',
            how='left'
        )
    else:
        for col in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'active_terms']:
            base_df[col] = None

    section07_elapsed_sec = _save_section_elapsed('07_base_merge', section07_start_ts)

    # 05b_active_retl: light query AFTER base merge (not inside heavy sql_trx) (not inside heavy sql_trx)
    # Point is active if >=1 of its terminals has n_amt_src > 1
    section05b_start_ts = time.perf_counter()
    if sa_df.empty or base_df.empty:
        base_df['active_retl_cnt'] = 0
        active_retl_df = pd.DataFrame(columns=['n_agr', 'active_retl_cnt'])
    else:
        sql_active_retl = f"""
        with fiid_rshb as (
          select distinct cast(fa.c_fiid as string) as c_fiid
          from ods_alpha.scd1_base24_fiids fa
          where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
        ),
        trx_base_raw as (
          select cast(t.n_trx as string) as n_trx, cast(t.c_nter as string) as c_nter, cast(t.n_amt_src as double) as n_amt_src
          from ods_alpha.scd1_trx t
          join fiid_rshb fr
            on fr.c_fiid = cast(t.c_fiid_acq as string)
          where cast(t.d_trx_orig as timestamp) >= cast('{month_start}' as timestamp)
            and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{month_end}' as date), 1) as timestamp)
            and t.c_nter is not null
            and coalesce(t.ods_deleted_flg, '0') <> '1'
            and t.c_trx_class = 'SA'
            and t.c_trx_type = 'S01'
            and coalesce(t.cf_trx_stat, '') <> 'R'
            and coalesce(cast(t.n_amt_src as double), 0.0) > 1
        ),
        trx_base as (
          select n_trx, max(c_nter) as c_nter
          from trx_base_raw
          group by n_trx
        ),
        active_nter as (
          select distinct
            cast(a.n_agr as string) as n_agr,
            tb.c_nter
          from ods_alpha.scd1_trx_acq a
          join trx_base tb on tb.n_trx = cast(a.n_trx as string)
          where a.n_agr is not null
            and coalesce(a.ods_deleted_flg, '0') <> '1'
        )
        select
          an.n_agr,
          count(distinct cast(p.c_nmrc as string)) as active_retl_cnt
        from active_nter an
        join ods_alpha.scd1_pos_terminals p
          on cast(p.c_nter as string) = an.c_nter
        where coalesce(p.ods_deleted_flg, '0') <> '1'
          and p.c_nmrc is not null
        group by an.n_agr
        """
        active_retl_df = _run_impala_fetch('05b_active_retl', sql_active_retl, mem_limit='8g')
        if active_retl_df is None:
            active_retl_df = pd.DataFrame(columns=['n_agr', 'active_retl_cnt'])
        if not active_retl_df.empty:
            active_retl_df['n_agr'] = active_retl_df['n_agr'].astype(str)
            active_retl_df['active_retl_cnt'] = pd.to_numeric(active_retl_df['active_retl_cnt'], errors='coerce')
            base_df = base_df.merge(active_retl_df[['n_agr', 'active_retl_cnt']], on='n_agr', how='left')
        else:
            base_df['active_retl_cnt'] = 0

    _arc = pd.to_numeric(base_df.get('active_retl_cnt'), errors='coerce').fillna(0)
    _rc = pd.to_numeric(base_df.get('retl_cnt'), errors='coerce').fillna(0)
    base_df['active_retl_cnt'] = _arc.clip(lower=0).combine(_rc, min)
    section05b_elapsed_sec = _save_section_elapsed('05b_active_retl', section05b_start_ts)
    _log_progress(
        f"05b_active_retl done: rows={0 if active_retl_df is None else len(active_retl_df)}, elapsed={section05b_elapsed_sec:.2f}s"
    )

    # 08_agr_fallback
    section08_start_ts = time.perf_counter()
    if 'agr_id' not in base_df.columns:
        base_df['agr_id'] = None
    if 'r2_id' not in base_df.columns:
        base_df['r2_id'] = None

    agr_before_mask = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
    base_df['agr_id_source'] = 'sa'
    fallback_mask = (~agr_before_mask) & base_df['r2_id'].notna() & (base_df['r2_id'].astype(str).str.strip() != '')
    base_df.loc[fallback_mask, 'agr_id'] = base_df.loc[fallback_mask, 'r2_id']
    base_df.loc[fallback_mask, 'agr_id_source'] = 'r2_fallback'

    section08_elapsed_sec = _save_section_elapsed('08_agr_fallback', section08_start_ts)

    # 08b_tariff_fix_map (fast mode: flat plan->commission map; actual tariff stays in section 09)
    section08b_start_ts = time.perf_counter()

    agr_id_scope = (
        base_df.get('agr_id', pd.Series(dtype=object))
        .map(normalize_agr_q1)
        .dropna()
        .astype(str)
        .str.strip()
    )
    agr_id_scope = sorted([x for x in agr_id_scope.unique().tolist() if x])

    sql_tariff_fix_map_full = """
        with tt_pairs as (
          select distinct
            tt.c_tariff_plan as c_tariff_plan_raw,
            tt.c_tariff as c_tariff_raw
          from ods.scd1_z_r2_tariff_tune tt
          where tt.c_tariff_plan is not null
        ),
        tf_agg as (
          select
            tf.id as c_tariff_raw,
            max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
          from ods.scd1_z_r2_tariff_fix tf
          group by tf.id
        )
        select
          cast(p.c_tariff_plan_raw as string) as c_tariff_plan,
          max(a.commission_monthly_fix) as commission_monthly_fix
        from tt_pairs p
        left join tf_agg a
          on p.c_tariff_raw = a.c_tariff_raw
        group by p.c_tariff_plan_raw
        """

    if use_fast_flat_08b_map and tariff_fix_map_flat_df is not None and len(tariff_fix_map_flat_df):
        tariff_fix_map_df = tariff_fix_map_flat_df.copy()
        map_source = 'preloaded_flat_map'
    elif use_fast_flat_08b_map and '_tariff_fix_map_runtime_cache_df' in globals() and len(globals()['_tariff_fix_map_runtime_cache_df']):
        tariff_fix_map_df = globals()['_tariff_fix_map_runtime_cache_df'].copy()
        map_source = 'runtime_cached_flat_map'
    elif use_fast_flat_08b_map:
        _log_progress('08b: preloaded flat map not found, building full flat map once in-month')
        tariff_fix_map_df = _run_impala_fetch(
            '08b_tariff_fix_map_full_once',
            sql_tariff_fix_map_full,
            mem_limit='8g',
            run_probe=True,
        )
        globals()['_tariff_fix_map_runtime_cache_df'] = tariff_fix_map_df.copy() if tariff_fix_map_df is not None else pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])
        map_source = 'rebuilt_flat_map'
    else:
        tariff_fix_map_df = _run_impala_fetch(
            '08b_tariff_fix_map_full_legacy_mode',
            sql_tariff_fix_map_full,
            mem_limit='8g',
            run_probe=True,
        )
        map_source = 'legacy_full_map'

    if tariff_fix_map_df is None:
        tariff_fix_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])

    if not tariff_fix_map_df.empty:
        tariff_fix_map_df['c_tariff_plan'] = tariff_fix_map_df['c_tariff_plan'].astype(str).str.strip()
        tariff_fix_map_df['commission_monthly_fix'] = pd.to_numeric(
            tariff_fix_map_df.get('commission_monthly_fix'), errors='coerce'
        )
        tariff_fix_map_df = (
            tariff_fix_map_df
            .dropna(subset=['c_tariff_plan'])
            .groupby('c_tariff_plan', as_index=False)['commission_monthly_fix']
            .max()
            .sort_values('c_tariff_plan')
            .reset_index(drop=True)
        )
        plan_scope = sorted([
            x for x in tariff_fix_map_df['c_tariff_plan'].dropna().astype(str).str.strip().unique().tolist()
            if x and x.lower() not in {'nan', 'none', 'null'}
        ])
    else:
        plan_scope = []

    section08b_elapsed_sec = _save_section_elapsed('08b_tariff_fix_map', section08b_start_ts)
    _log_progress(
        f"08b summary: mode={'fast_flat' if use_fast_flat_08b_map else 'legacy_full'}, "
        f"source={map_source}, "
        f"agr_id_scope={len(agr_id_scope):,}, "
        f"plan_scope={len(plan_scope):,}, "
        f"tariff_fix_map_rows={len(tariff_fix_map_df):,}, "
        f"elapsed_sec={section08b_elapsed_sec}"
    )

    # 09_actual_tariff_by_agr
    section09_start_ts = time.perf_counter()
    base_df['agr_id_key'] = base_df['agr_id'].map(normalize_agr_q1)
    base_agr_keys_df = (
        base_df[['agr_id_key']]
        .dropna()
        .drop_duplicates()
        .rename(columns={'agr_id_key': 'agr_id'})
    )

    base_agr_scope = []
    actual_plan_scope = []
    actual_tariff_elapsed_sec = 0.0
    commission_map_elapsed_sec = 0.0
    actual_tariff_fetch_mode = 'empty_scope'
    commission_map_source = 'empty_scope'

    def _build_sql_actual_tariff_by_agr(agr_scope):
        agr_id_in = _in_sql_list(agr_scope)
        return f"""
        with agr_actual as (
          select
            cast(a.abs_agr_id as string) as agr_id,
            cast(a.n_agr as string) as n_agr_actual,
            cast(a.c_agr_number as string) as contract_number_acq,
            cast(a.d_valid_from as date) as d_valid_from_actual,
            cast(a.d_valid_to as date) as d_valid_to_actual,
            row_number() over (
              partition by cast(a.abs_agr_id as string)
              order by cast(a.d_valid_from as date) desc, cast(a.n_agr as string) desc
            ) as rn
          from ods_alpha.scd1_agreements a
          where cast(a.abs_agr_id as string) in ({agr_id_in})
            and upper(trim(cast(a.acq_class as string))) = 'SA'
            and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
            and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_end}' as date))
            and coalesce(a.ods_deleted_flg, '0') <> '1'
        ),
        merchant_one as (
          select
            cast(m.id as string) as agr_id,
            cast(m.c_tariff_plan as string) as c_tariff_plan,
            row_number() over (
              partition by cast(m.id as string)
              order by cast(m.c_tariff_plan as string) desc
            ) as rn
          from ods.scd1_z_r2_ip_merchants m
          where cast(m.id as string) in ({agr_id_in})
            and coalesce(m.ods_deleted_flg, '0') <> '1'
        )
        select
          m.agr_id,
          aa.n_agr_actual,
          aa.contract_number_acq,
          aa.d_valid_from_actual,
          aa.d_valid_to_actual,
          m.c_tariff_plan,
          cast(tp.c_name as string) as tariff_name_actual
        from merchant_one m
        left join agr_actual aa
          on aa.agr_id = m.agr_id
         and aa.rn = 1
        left join ods.scd1_z_r2_tariff_plan tp
          on cast(tp.id as string) = m.c_tariff_plan
        where m.rn = 1
        """

    if len(base_agr_keys_df):
        base_agr_scope = sorted(
            base_agr_keys_df['agr_id']
            .dropna()
            .astype(str)
            .str.strip()
            .loc[lambda s: s != '']
            .unique()
            .tolist()
        )

        actual_tariff_fetch_start_ts = time.perf_counter()
        if len(base_agr_scope) > int(section09_chunk_threshold):
            scope_chunks = _split_scope(base_agr_scope, section09_chunk_size)
            actual_tariff_fetch_mode = f"chunked_{len(scope_chunks)}"
            _log_progress(
                f"09_actual_tariff_by_agr: chunked fetch enabled, agr_scope={len(base_agr_scope):,}, "
                f"chunk_size={section09_chunk_size}, chunks={len(scope_chunks):,}"
            )
            actual_tariff_parts = []
            for idx, agr_chunk in enumerate(scope_chunks, start=1):
                chunk_df = _run_impala_fetch(
                    f"09_actual_tariff_by_agr_chunk_{idx}_of_{len(scope_chunks)}",
                    _build_sql_actual_tariff_by_agr(agr_chunk),
                    mem_limit='12g',
                    run_probe=(idx == 1),
                )
                if chunk_df is not None and len(chunk_df):
                    actual_tariff_parts.append(chunk_df)

            if actual_tariff_parts:
                actual_tariff_df = pd.concat(actual_tariff_parts, ignore_index=True)
            else:
                actual_tariff_df = pd.DataFrame(columns=[
                    'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
                    'c_tariff_plan', 'tariff_name_actual'
                ])
        else:
            actual_tariff_fetch_mode = 'single_shot'
            actual_tariff_df = _run_impala_fetch(
                '09_actual_tariff_by_agr',
                _build_sql_actual_tariff_by_agr(base_agr_scope),
                mem_limit='12g'
            )

        actual_tariff_elapsed_sec = round(time.perf_counter() - actual_tariff_fetch_start_ts, 2)
    else:
        actual_tariff_df = pd.DataFrame(columns=[
            'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
            'c_tariff_plan', 'tariff_name_actual'
        ])

    if actual_tariff_df is None:
        actual_tariff_df = pd.DataFrame(columns=[
            'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
            'c_tariff_plan', 'tariff_name_actual'
        ])

    if not actual_tariff_df.empty:
        for c in ['agr_id', 'n_agr_actual', 'contract_number_acq', 'c_tariff_plan', 'tariff_name_actual']:
            actual_tariff_df[c] = actual_tariff_df[c].astype(str).str.strip()
        actual_tariff_df = actual_tariff_df.merge(base_agr_keys_df, on='agr_id', how='inner')

    actual_tariff_df = actual_tariff_df.rename(columns={'agr_id': 'agr_id_key'})

    if 'c_tariff_plan' not in actual_tariff_df.columns:
        actual_tariff_df['c_tariff_plan'] = None

    actual_comm_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])
    if not actual_tariff_df.empty:
        actual_plan_scope = sorted([
            x for x in actual_tariff_df['c_tariff_plan'].dropna().astype(str).str.strip().unique().tolist()
            if x and x.lower() not in {'nan', 'none', 'null'}
        ])

        if actual_plan_scope:
            commission_map_fetch_start_ts = time.perf_counter()
            if tariff_fix_map_df is not None and len(tariff_fix_map_df):
                commission_map_source = '08b_cached_scope_filter'
                actual_comm_map_df = tariff_fix_map_df[
                    tariff_fix_map_df['c_tariff_plan'].astype(str).str.strip().isin(set(actual_plan_scope))
                ][['c_tariff_plan', 'commission_monthly_fix']].copy()
            else:
                commission_map_source = '09_sql_scope'
                plan_in = _in_sql_list(actual_plan_scope)
                sql_actual_commission_by_plan = f"""
                with tt_pairs as (
                  select distinct
                    tt.c_tariff_plan as c_tariff_plan_raw,
                    tt.c_tariff as c_tariff_raw
                  from ods.scd1_z_r2_tariff_tune tt
                  where cast(tt.c_tariff_plan as string) in ({plan_in})
                ),
                tf_agg as (
                  select
                    tf.id as c_tariff_raw,
                    max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
                  from ods.scd1_z_r2_tariff_fix tf
                  group by tf.id
                )
                select
                  cast(p.c_tariff_plan_raw as string) as c_tariff_plan,
                  max(a.commission_monthly_fix) as commission_monthly_fix
                from tt_pairs p
                left join tf_agg a
                  on p.c_tariff_raw = a.c_tariff_raw
                group by p.c_tariff_plan_raw
                """

                actual_comm_map_df = _run_impala_fetch(
                    '09_commission_map_by_actual_plan',
                    sql_actual_commission_by_plan,
                    mem_limit='8g',
                    run_probe=True,
                )

            commission_map_elapsed_sec = round(time.perf_counter() - commission_map_fetch_start_ts, 2)

    if actual_comm_map_df is None:
        actual_comm_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])

    if not actual_comm_map_df.empty:
        actual_comm_map_df['c_tariff_plan'] = actual_comm_map_df['c_tariff_plan'].astype(str).str.strip()
        actual_comm_map_df['commission_monthly_fix'] = pd.to_numeric(
            actual_comm_map_df.get('commission_monthly_fix'), errors='coerce'
        )
        actual_comm_map_df = (
            actual_comm_map_df
            .dropna(subset=['c_tariff_plan'])
            .groupby('c_tariff_plan', as_index=False)['commission_monthly_fix']
            .max()
        )
        actual_tariff_df = actual_tariff_df.merge(actual_comm_map_df, on='c_tariff_plan', how='left')
    else:
        actual_tariff_df['commission_monthly_fix'] = None

    actual_tariff_df['commission_monthly_actual'] = pd.to_numeric(
        actual_tariff_df.get('commission_monthly_fix'), errors='coerce'
    )

    if 'commission_monthly_fix' in actual_tariff_df.columns:
        actual_tariff_df = actual_tariff_df.drop(columns=['commission_monthly_fix'])

    section09_debug = {
        'report_month': report_month_label,
        'agr_scope_count': int(len(base_agr_scope)),
        'actual_tariff_rows': int(len(actual_tariff_df)),
        'actual_plan_scope_count': int(len(actual_plan_scope)),
        'actual_tariff_fetch_mode': actual_tariff_fetch_mode,
        'actual_tariff_elapsed_sec': float(actual_tariff_elapsed_sec),
        'commission_map_elapsed_sec': float(commission_map_elapsed_sec),
        'commission_map_source': commission_map_source,
    }
    globals().setdefault('section09_debug_by_month', {})
    globals()['section09_debug_by_month'][report_month_label] = section09_debug

    _log_progress(
        f"09 baseline: agr_scope={len(base_agr_scope):,}, mode={actual_tariff_fetch_mode}, "
        f"actual_rows={len(actual_tariff_df):,}, actual_plan_scope={len(actual_plan_scope):,}, "
        f"elapsed_09a={actual_tariff_elapsed_sec}s, elapsed_09b={commission_map_elapsed_sec}s, "
        f"comm_map_source={commission_map_source}"
    )

    section09_elapsed_sec = _save_section_elapsed('09_actual_tariff_by_agr', section09_start_ts)

    # 10_apply_tariff_fix_and_formulas -> final_df
    section10_start_ts = time.perf_counter()
    required_tariff_cols = [
        'agr_id_key', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual',
        'd_valid_to_actual', 'c_tariff_plan', 'tariff_name_actual', 'commission_monthly_actual'
    ]
    for col in required_tariff_cols:
        if col not in actual_tariff_df.columns:
            actual_tariff_df[col] = None

    merge_payload_cols = [c for c in required_tariff_cols if c != 'agr_id_key']
    drop_before_merge = []
    for c in merge_payload_cols:
        for c_try in [c, f'{c}_x', f'{c}_y']:
            if c_try in base_df.columns:
                drop_before_merge.append(c_try)
    if drop_before_merge:
        base_df = base_df.drop(columns=sorted(set(drop_before_merge)))

    base_df = base_df.merge(
        actual_tariff_df[required_tariff_cols],
        on='agr_id_key',
        how='left'
    )

    base_df['tariff_name'] = base_df['tariff_name_actual'].where(
        base_df['tariff_name_actual'].notna() & (base_df['tariff_name_actual'].astype(str).str.strip() != ''),
        base_df['tariff_name_legacy']
    )
    base_df['tariff_source'] = base_df['tariff_name_actual'].apply(
        lambda x: 'agr_id_actual' if pd.notna(x) and str(x).strip() else 'legacy_cft'
    )
    base_df['tariff_short'] = base_df['tariff_name'].apply(lambda x: build_tariff_short(x, source='lake'))

    commission_monthly_actual_num = pd.to_numeric(base_df.get('commission_monthly_actual'), errors='coerce')
    commission_monthly_legacy_num = pd.to_numeric(base_df.get('commission_monthly_legacy'), errors='coerce')
    commission_monthly_plan_fallback_num = pd.Series(np.nan, index=base_df.index, dtype='float64')

    if tariff_fix_map_df is not None and len(tariff_fix_map_df):
        tariff_fix_map_series = (
            tariff_fix_map_df.assign(c_tariff_plan=lambda d: d['c_tariff_plan'].astype(str).str.strip())
            .dropna(subset=['c_tariff_plan'])
            .drop_duplicates(subset=['c_tariff_plan'], keep='first')
            .set_index('c_tariff_plan')['commission_monthly_fix']
        )
        tariff_plan_for_fallback = base_df.get('c_tariff_plan', pd.Series(index=base_df.index, dtype=object))
        commission_monthly_plan_fallback_num = pd.to_numeric(
            tariff_plan_for_fallback.astype(str).str.strip().map(tariff_fix_map_series),
            errors='coerce'
        )

    if use_legacy_commission_monthly_only:
        commission_mode_label = 'legacy_only_flag'
        base_df['commission_monthly'] = commission_monthly_legacy_num
    else:
        commission_mode_label = 'actual_then_08b_then_legacy'
        base_df['commission_monthly'] = commission_monthly_actual_num.where(
            commission_monthly_actual_num.notna(),
            commission_monthly_plan_fallback_num.where(
                commission_monthly_plan_fallback_num.notna(),
                commission_monthly_legacy_num
            )
        )

    commission_actual_fill_pct = round(100.0 * commission_monthly_actual_num.notna().mean(), 2)
    commission_plan_fill_pct = round(100.0 * commission_monthly_plan_fallback_num.notna().mean(), 2)
    commission_legacy_fill_pct = round(100.0 * commission_monthly_legacy_num.notna().mean(), 2)
    commission_zero_pct = round(100.0 * pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0).eq(0).mean(), 2)

    _log_progress(
        f"09 fill-rates ({commission_mode_label}): actual={commission_actual_fill_pct}%, "
        f"plan_fallback_08b={commission_plan_fill_pct}%, "
        f"legacy={commission_legacy_fill_pct}%, "
        f"zero_final={commission_zero_pct}%"
    )

    globals().setdefault('section09_fill_debug_by_month', {})
    globals()['section09_fill_debug_by_month'][report_month_label] = {
        'report_month': report_month_label,
        'commission_mode': commission_mode_label,
        'actual_fill_pct': float(commission_actual_fill_pct),
        'plan_fallback_fill_pct': float(commission_plan_fill_pct),
        'legacy_fill_pct': float(commission_legacy_fill_pct),
        'zero_final_pct': float(commission_zero_pct),
    }


    # 10m_mpos_commission_monthly: REPLACE cascade with MPOS_RENT.n_amt
    # Keep actual/08b/legacy numerics above for diagnostics only; final value is MPOS.
    section10m_start_ts = time.perf_counter()
    try:
        mpos_comm_df = _run_impala_fetch(
            '10m_mpos_commission_monthly',
            build_mpos_commission_monthly_sql(month_start, month_end),
            mem_limit='8g',
        )
    except Exception as exc_mpos:
        _log_progress(f"10m MPOS commission failed: {type(exc_mpos).__name__}: {exc_mpos}; fill zeros")
        mpos_comm_df = pd.DataFrame(columns=['inn', 'agr_id', 'commission_monthly_mpos'])

    cascade_before = pd.to_numeric(base_df.get('commission_monthly'), errors='coerce')
    base_df = merge_mpos_commission_monthly(base_df, mpos_comm_df)
    mpos_hit_pct = round(
        100.0 * pd.to_numeric(base_df.get('commission_monthly_mpos'), errors='coerce').notna().mean(),
        2,
    )
    mpos_nonzero_pct = round(
        100.0 * pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0).ne(0).mean(),
        2,
    )
    cascade_nonzero_pct = round(100.0 * cascade_before.fillna(0).ne(0).mean(), 2) if cascade_before is not None else None
    _log_progress(
        f"10m MPOS commission: keys={len(mpos_comm_df) if mpos_comm_df is not None else 0:,}, "
        f"hit_pct={mpos_hit_pct}%, nonzero_pct={mpos_nonzero_pct}%, "
        f"old_cascade_nonzero_pct={cascade_nonzero_pct}%"
    )
    globals().setdefault('section10m_mpos_debug_by_month', {})
    globals()['section10m_mpos_debug_by_month'][report_month_label] = {
        'report_month': report_month_label,
        'mpos_keys': int(len(mpos_comm_df) if mpos_comm_df is not None else 0),
        'hit_pct': float(mpos_hit_pct),
        'nonzero_pct': float(mpos_nonzero_pct),
        'old_cascade_nonzero_pct': float(cascade_nonzero_pct) if cascade_nonzero_pct is not None else None,
    }
    _save_section_elapsed('10m_mpos_commission_monthly', section10m_start_ts)

    commission_from_ops_num = pd.to_numeric(base_df.get('commission_from_ops'), errors='coerce').fillna(0)
    commission_monthly_num = pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0)
    int_component_num = pd.to_numeric(base_df.get('int_component'), errors='coerce').fillna(0)
    amortization_num = pd.to_numeric(base_df.get('amortization'), errors='coerce').fillna(0)
    retl_cnt_num = pd.to_numeric(base_df.get('retl_cnt'), errors='coerce').fillna(0)

    base_df['commission_total'] = commission_from_ops_num + commission_monthly_num
    base_df['aur'] = retl_cnt_num * 1926
    base_df['chod'] = base_df['commission_total'] + int_component_num
    base_df['fin_result'] = base_df['chod'] - pd.to_numeric(base_df['aur'], errors='coerce').fillna(0) - amortization_num
    base_df['report_month'] = report_month_label
    base_df['snapshot_month_start'] = snapshot_month_start

    mvp_columns = [
        'report_month', 'snapshot_month_start', 'inn', 'company_name',
        'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
        'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
        'cdi_id', 'ssp_ocrm', 'cft_id', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code',
        'tariff_name', 'tariff_short', 'tariff_source',
        'retl_cnt', 'active_retl_cnt', 'term_cnt', 'active_terms', 'active_term_cnt', 'trx_cnt', 'trx_sum',
        'commission_from_ops', 'commission_monthly', 'commission_monthly_source', 'commission_monthly_mpos', 'int_component', 'commission_total',
        'aur', 'amortization', 'chod', 'fin_result'
    ]

    for col in mvp_columns:
        if col not in base_df.columns:
            base_df[col] = None

    for col in [
        'trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component',
        'commission_total', 'aur', 'amortization', 'chod', 'fin_result'
    ]:
        base_df[col] = base_df[col].map(to_money_decimal_2_or_none)

    for col in ['retl_cnt', 'active_retl_cnt', 'term_cnt', 'active_terms', 'active_term_cnt', 'trx_cnt']:
        base_df[col] = pd.to_numeric(base_df[col], errors='coerce').astype('Int64')


    # 10b_client_product_flags -> join by inn on month_end
    section10b_start_ts = time.perf_counter()
    try:
        products_long_df = _run_impala_fetch(
            '10b_client_product_flags',
            build_client_products_long_sql(month_end),
            mem_limit='8g',
        )
        products_wide_df = pivot_client_products_wide(products_long_df)
    except Exception as exc_products:
        _log_progress(
            f"10b product flags failed: {type(exc_products).__name__}: {exc_products}; fill zeros"
        )
        products_wide_df = pd.DataFrame(columns=['inn'] + PRODUCT_FLAG_COLUMNS)

    rows_before_products = len(base_df)
    base_df = merge_product_flags(base_df, products_wide_df)
    product_flag_cols = [c for c in PRODUCT_FLAG_COLUMNS if c in base_df.columns]
    extra_product_cols = [
        c for c in products_wide_df.columns
        if c != 'inn'
        and c not in PRODUCT_FLAG_COLUMNS
        and c not in EXCLUDED_PRODUCT_COLUMNS
        and c in base_df.columns
    ]
    product_flag_cols = product_flag_cols + extra_product_cols
    base_df['recommended_products'] = build_recommended_products(base_df)
    mvp_columns = mvp_columns + product_flag_cols + ['recommended_products']

    if len(base_df) != rows_before_products:
        raise RuntimeError(
            f'[{report_month_label}] product flags join changed row count: '
            f'{rows_before_products} -> {len(base_df)}'
        )

    _log_progress(
        f"10b product flags: product_inns={len(products_wide_df):,}, flag_cols={len(product_flag_cols)}, recommended_nonempty={int((base_df['recommended_products'].astype(str).str.len() > 0).sum()):,}"
    )
    _save_section_elapsed('10b_client_product_flags', section10b_start_ts)

    final_df = base_df[mvp_columns].copy()

    if final_df is None:
        raise RuntimeError(f'[{report_month_label}] final_df is None')

    section10_elapsed_sec = _save_section_elapsed('10_apply_tariff_fix_and_formulas', section10_start_ts)

    total_elapsed_sec = round(time.perf_counter() - month_total_start_ts, 2)
    section_order = [
        '01_sa_perimeter',
        '02_cdi_map',
        '03_cft_map',
        '04_operational_metrics',
        '05_transaction_metrics',
        '06_r2_legacy_attrs',
        '07_base_merge',
        '08_agr_fallback',
        '08b_tariff_fix_map',
        '09_actual_tariff_by_agr',
        '10_apply_tariff_fix_and_formulas',
        '10m_mpos_commission_monthly',
        '10b_client_product_flags',
    ]
    section_timing_df = pd.DataFrame([
        {'section': k, 'elapsed_sec': section_elapsed_tracker[k]}
        for k in section_order
        if k in section_elapsed_tracker
    ])

    _log_progress(f"final_df ready: rows={len(final_df):,}, total_elapsed={total_elapsed_sec}s")
    if len(section_timing_df):
        print('Section timing (01->10b):')
        display(section_timing_df)
    return final_df


In [ ]:
METRICS_ORDER = [
    'unique_inn',
    'retl_cnt',
    'term_cnt',
    'trx_cnt',
    'trx_sum',
    'commission_from_ops',
    'commission_monthly',
    'commission_total',
    'int_component',
    'chod',
    'aur',
    'amortization',
    'fin_result',
]


def build_lake_agg(final_df):
    if final_df is None or final_df.empty:
        return pd.DataFrame(
            columns=[
                'inn_key', 'agr_id_key', 'retl_cnt_lake', 'term_cnt_lake', 'trx_cnt_lake',
                'trx_sum_lake', 'commission_from_ops_lake', 'commission_monthly_lake',
                'commission_total_lake', 'int_component_lake', 'chod_lake', 'aur_lake', 'amortization_lake', 'fin_result_lake'
            ]
        )

    lk = final_df.copy()
    lk['inn_key'] = lk['inn'].apply(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].apply(normalize_agr_q1)

    lk['retl_cnt_lake'] = pd.to_numeric(lk['retl_cnt'], errors='coerce')
    lk['term_cnt_lake'] = pd.to_numeric(lk['term_cnt'], errors='coerce')
    lk['trx_cnt_lake'] = pd.to_numeric(lk['trx_cnt'], errors='coerce')
    lk['trx_sum_lake'] = pd.to_numeric(lk['trx_sum'], errors='coerce')
    lk['commission_from_ops_lake'] = pd.to_numeric(lk['commission_from_ops'], errors='coerce')
    lk['commission_monthly_lake'] = pd.to_numeric(lk['commission_monthly'], errors='coerce')
    lk['commission_total_lake'] = pd.to_numeric(lk['commission_total'], errors='coerce')
    lk['int_component_lake'] = pd.to_numeric(lk['int_component'], errors='coerce')
    lk['chod_lake'] = pd.to_numeric(lk['chod'], errors='coerce')
    lk['aur_lake'] = pd.to_numeric(lk.get('aur'), errors='coerce')
    lk['amortization_lake'] = pd.to_numeric(lk.get('amortization'), errors='coerce')
    lk['fin_result_lake'] = pd.to_numeric(lk.get('fin_result'), errors='coerce')

    lake_agg = (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_lake': 'max',
              'term_cnt_lake': 'max',
              'trx_cnt_lake': 'max',
              'trx_sum_lake': 'max',
              'commission_from_ops_lake': 'max',
              'commission_monthly_lake': 'max',
              'commission_total_lake': 'max',
              'int_component_lake': 'max',
              'chod_lake': 'max',
              'aur_lake': 'max',
              'amortization_lake': 'max',
              'fin_result_lake': 'max',
          })
    )
    return lake_agg


def load_excel_agg(excel_path, excel_header=0):
    ex = pd.read_excel(excel_path, header=excel_header)

    col_map = {
        'inn_col': ['ИНН', 'inn', 'c_inn'],
        'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
        'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
        'term_col': ['Кол-во терминалов', 'Количество терминалов'],
        'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
        'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
        'comm_ops_col': ['Комиссия эквайринга', 'Комиссия (% с операций)', 'Комиссия \n(% с операций)', 'Комиссия % с операций'],
        'comm_monthly_col': ['Комиссия в месяц', 'Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)', 'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)'],
        'comm_total_col': [
            'Общая комиссия',
            'Комиссия общая',
            'Итого комиссия',
            'Итоговая комиссия',
            'Комиссия эквайринга',
            'Коммиссия эквайринга',
        ],
        'int_component_col': ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)'],
        'chod_col': ['ЧОД'],
        'aur_col': ['АУР', 'AUR', 'Aur', 'Аур'],
        'amortization_col': [
            'Амортизация', 'Аморт', 'Амортизация терминалов',
            'amortization', 'Amortization', 'Амортизация, руб',
        ],
        'fin_result_col': [
            'Фин. Рез.', 'Фин.Рез.', 'Фин.рез.', 'Фин. рез.',
            'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат',
            'fin_result', 'Fin.Res.', 'FinRes',
        ],
    }

    resolved = {k: pick_col_robust(ex.columns, v) for k, v in col_map.items()}
    optional_excel_cols = {'aur_col', 'amortization_col', 'fin_result_col'}
    missing = [k for k, v in resolved.items() if v is None and k not in optional_excel_cols]
    if missing:
        raise ValueError(f'Не найдены колонки Excel: {missing}. Доступные: {list(ex.columns)}')

    ex['inn_key'] = ex[resolved['inn_col']].apply(normalize_inn_q1)
    ex['agr_id_key'] = ex[resolved['agr_col']].apply(normalize_agr_q1)

    ex['retl_cnt_excel'] = pd.to_numeric(ex[resolved['retl_col']], errors='coerce')
    ex['term_cnt_excel'] = pd.to_numeric(ex[resolved['term_col']], errors='coerce')
    ex['trx_cnt_excel'] = pd.to_numeric(ex[resolved['trx_cnt_col']], errors='coerce')
    ex['trx_sum_excel'] = to_num_series(ex[resolved['trx_sum_col']])
    ex['commission_from_ops_excel'] = to_num_series(ex[resolved['comm_ops_col']])
    ex['commission_monthly_excel'] = to_num_series(ex[resolved['comm_monthly_col']])
    ex['commission_total_excel'] = to_num_series(ex[resolved['comm_total_col']])
    ex['int_component_excel'] = to_num_series(ex[resolved['int_component_col']])
    ex['chod_excel'] = to_num_series(ex[resolved['chod_col']])
    if resolved.get('aur_col') is not None:
        ex['aur_excel'] = to_num_series(ex[resolved['aur_col']])
    else:
        ex['aur_excel'] = np.nan
    if resolved.get('amortization_col') is not None:
        ex['amortization_excel'] = to_num_series(ex[resolved['amortization_col']])
    else:
        ex['amortization_excel'] = np.nan
    if resolved.get('fin_result_col') is not None:
        ex['fin_result_excel'] = to_num_series(ex[resolved['fin_result_col']])
    else:
        ex['fin_result_excel'] = np.nan

    ex_agg = (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_excel': 'max',
              'term_cnt_excel': 'max',
              'trx_cnt_excel': 'max',
              'trx_sum_excel': 'sum',
              'commission_from_ops_excel': 'sum',
              'commission_monthly_excel': 'max',
              'commission_total_excel': 'sum',
              'int_component_excel': 'sum',
              'chod_excel': 'sum',
              'aur_excel': 'max',
              'amortization_excel': 'sum',
              'fin_result_excel': 'sum',
          })
    )
    return ex_agg, resolved


def calc_lake_totals(lake_agg):
    if lake_agg is None or lake_agg.empty:
        return {m: 0.0 if m != 'unique_inn' else 0 for m in METRICS_ORDER}

    return {
        'unique_inn': int(lake_agg['inn_key'].nunique()),
        'retl_cnt': float(lake_agg['retl_cnt_lake'].fillna(0).sum()),
        'term_cnt': float(lake_agg['term_cnt_lake'].fillna(0).sum()),
        'trx_cnt': float(lake_agg['trx_cnt_lake'].fillna(0).sum()),
        'trx_sum': float(lake_agg['trx_sum_lake'].fillna(0).sum()),
        'commission_from_ops': float(lake_agg['commission_from_ops_lake'].fillna(0).sum()),
        'commission_monthly': float(lake_agg['commission_monthly_lake'].fillna(0).sum()),
        'commission_total': float(lake_agg['commission_total_lake'].fillna(0).sum()),
        'int_component': float(lake_agg['int_component_lake'].fillna(0).sum()),
        'chod': float(lake_agg['chod_lake'].fillna(0).sum()),
        'aur': float(lake_agg['aur_lake'].fillna(0).sum()),
        'amortization': float(lake_agg['amortization_lake'].fillna(0).sum()),
        'fin_result': float(lake_agg['fin_result_lake'].fillna(0).sum()),
    }


def calc_excel_totals(ex_agg):
    if ex_agg is None or ex_agg.empty:
        return {m: 0.0 if m != 'unique_inn' else 0 for m in METRICS_ORDER}

    return {
        'unique_inn': int(ex_agg['inn_key'].nunique()),
        'retl_cnt': float(ex_agg['retl_cnt_excel'].fillna(0).sum()),
        'term_cnt': float(ex_agg['term_cnt_excel'].fillna(0).sum()),
        'trx_cnt': float(ex_agg['trx_cnt_excel'].fillna(0).sum()),
        'trx_sum': float(ex_agg['trx_sum_excel'].fillna(0).sum()),
        'commission_from_ops': float(ex_agg['commission_from_ops_excel'].fillna(0).sum()),
        'commission_monthly': float(ex_agg['commission_monthly_excel'].fillna(0).sum()),
        'commission_total': float(ex_agg['commission_total_excel'].fillna(0).sum()),
        'int_component': float(ex_agg['int_component_excel'].fillna(0).sum()),
        'chod': float(ex_agg['chod_excel'].fillna(0).sum()),
        'aur': (
            float(ex_agg['aur_excel'].fillna(0).sum())
            if ex_agg['aur_excel'].notna().any()
            else np.nan
        ),
        'amortization': (
            float(ex_agg['amortization_excel'].fillna(0).sum())
            if ex_agg['amortization_excel'].notna().any()
            else np.nan
        ),
        'fin_result': (
            float(ex_agg['fin_result_excel'].fillna(0).sum())
            if ex_agg['fin_result_excel'].notna().any()
            else np.nan
        ),
    }


def build_monthly_compare(report_month_label, final_df, excel_path=None, excel_header=0):
    lake_agg = build_lake_agg(final_df)
    lake_totals = calc_lake_totals(lake_agg)

    if excel_path is None:
        rows = []
        for metric in METRICS_ORDER:
            rows.append({
                'report_month': report_month_label,
                'metric': metric,
                'lake_value': lake_totals[metric],
                'excel_value': np.nan,
                'delta_lake_minus_excel': np.nan,
                'divergence_pct': np.nan,
                'reference_status': 'no_excel_reference',
                'excel_path': None,
            })
        month_df = pd.DataFrame(rows)
        return month_df, lake_agg, None

    if not Path(excel_path).exists():
        raise FileNotFoundError(f'Excel reference not found for {report_month_label}: {excel_path}')

    ex_agg, resolved = load_excel_agg(excel_path, excel_header=excel_header)
    excel_totals = calc_excel_totals(ex_agg)

    rows = []
    for metric in METRICS_ORDER:
        lake_value = lake_totals[metric]
        excel_value = excel_totals[metric]
        delta = lake_value - excel_value
        rows.append({
            'report_month': report_month_label,
            'metric': metric,
            'lake_value': lake_value,
            'excel_value': excel_value,
            'delta_lake_minus_excel': delta,
            'divergence_pct': safe_divergence_pct(delta, excel_value),
            'reference_status': 'has_excel_reference',
            'excel_path': excel_path,
        })

    month_df = pd.DataFrame(rows)
    return month_df, lake_agg, {'excel_agg': ex_agg, 'resolved_columns': resolved}


In [ ]:
final_df_by_month = {}
monthly_compare_parts = []
lake_agg_by_month = {}
excel_meta_by_month = {}

monthly_checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Force rebuild after new amortization: wipe / skip stale monthly checkpoints
if force_recompute_final_df and wipe_checkpoints_on_force and monthly_checkpoint_dir.exists():
    removed = 0
    for p in monthly_checkpoint_dir.glob('final_df_*'):
        p.unlink(missing_ok=True)
        removed += 1
    for p in monthly_checkpoint_dir.glob('monthly_compare_*'):
        p.unlink(missing_ok=True)
        removed += 1
    if monthly_run_state_path.exists():
        monthly_run_state_path.unlink(missing_ok=True)
        removed += 1
    print(f'[force_recompute] wiped {removed} checkpoint file(s) in {monthly_checkpoint_dir}')
elif force_recompute_final_df:
    print('[force_recompute] checkpoints will be ignored (wipe_checkpoints_on_force=False)')



def _load_run_state(state_path):
    default_state = {
        'period_start': period_start,
        'period_end': period_end,
        'completed_months': [],
        'months': {},
    }
    if not state_path.exists():
        return default_state

    try:
        with open(state_path, 'r', encoding='utf-8') as f_in:
            loaded = json.load(f_in)
        if not isinstance(loaded, dict):
            return default_state
        loaded.setdefault('period_start', period_start)
        loaded.setdefault('period_end', period_end)
        loaded.setdefault('completed_months', [])
        loaded.setdefault('months', {})
        return loaded
    except Exception:
        return default_state


def _save_run_state(state_path, state_payload):
    with open(state_path, 'w', encoding='utf-8') as f_out:
        json.dump(state_payload, f_out, ensure_ascii=False, indent=2)


run_state = _load_run_state(monthly_run_state_path)
run_state['period_start'] = period_start
run_state['period_end'] = period_end

for month_ts in period_months:
    report_month = month_ts.strftime('%Y-%m-%d')
    report_month_label = month_ts.strftime('%Y-%m')
    excel_path = excel_reference_by_month.get(report_month_label)
    excel_header_current = int(excel_header_by_month.get(report_month_label, excel_header))

    month_key = report_month_label.replace('-', '_')
    month_final_parquet_path = monthly_checkpoint_dir / f'final_df_{month_key}.parquet'
    month_final_csv_path = monthly_checkpoint_dir / f'final_df_{month_key}.csv'
    month_compare_csv_path = monthly_checkpoint_dir / f'monthly_compare_{month_key}.csv'

    checkpoint_loaded = False
    final_df_month = None
    month_compare_df = None
    lake_agg_df = None
    excel_meta = None
    final_df_format = None
    final_df_checkpoint_path = None

    if (not force_recompute_final_df) and (month_final_parquet_path.exists() or month_final_csv_path.exists()):
        try:
            if month_final_parquet_path.exists():
                final_df_month = pd.read_parquet(month_final_parquet_path)
                final_df_format = 'parquet'
                final_df_checkpoint_path = month_final_parquet_path
            else:
                final_df_month = pd.read_csv(month_final_csv_path)
                final_df_format = 'csv'
                final_df_checkpoint_path = month_final_csv_path

            if month_compare_csv_path.exists():
                month_compare_df = pd.read_csv(month_compare_csv_path)
                found_metrics = set(
                    month_compare_df.get('metric', pd.Series(dtype=object)).astype(str).tolist()
                )
                # Stale checkpoint from before aur/fin_result were added
                if not set(METRICS_ORDER).issubset(found_metrics):
                    print(
                        f'[{report_month_label}] compare_checkpoint_stale '
                        f'(missing {sorted(set(METRICS_ORDER) - found_metrics)}) — rebuild'
                    )
                    month_compare_df, lake_agg_df, excel_meta = build_monthly_compare(
                        report_month_label=report_month_label,
                        final_df=final_df_month,
                        excel_path=excel_path,
                        excel_header=excel_header_current,
                    )
                    month_compare_df.to_csv(month_compare_csv_path, index=False, encoding='utf-8-sig')
            else:
                month_compare_df, lake_agg_df, excel_meta = build_monthly_compare(
                    report_month_label=report_month_label,
                    final_df=final_df_month,
                    excel_path=excel_path,
                    excel_header=excel_header_current,
                )
                month_compare_df.to_csv(month_compare_csv_path, index=False, encoding='utf-8-sig')

            if final_df_month is None or final_df_month.empty:
                raise RuntimeError('empty_final_df_checkpoint')

            if 'report_month' in final_df_month.columns:
                month_values = set(final_df_month['report_month'].dropna().astype(str).unique().tolist())
                if month_values and report_month_label not in month_values:
                    raise RuntimeError('checkpoint_month_mismatch')

            checkpoint_loaded = True
            print(
                f'[{report_month_label}] checkpoint_loaded: '
                f'final_df_rows={len(final_df_month):,}, '
                f'compare_rows={len(month_compare_df):,}, format={final_df_format}'
            )
        except Exception as exc_checkpoint:
            checkpoint_loaded = False
            final_df_month = None
            month_compare_df = None
            print(f'[{report_month_label}] checkpoint_corrupted_recompute: {type(exc_checkpoint).__name__}')

    if not checkpoint_loaded:
        month_start_ts = time.perf_counter()

        final_df_month = compute_final_df_for_month(
            report_month=report_month,
            imp=imp,
            tariff_fix_map_flat_df=tariff_fix_map_flat_df,
            use_fast_flat_08b_map=use_fast_flat_08b_map,
            use_legacy_commission_monthly_only=use_legacy_commission_monthly_only,
            section06_chunk_threshold=section06_chunk_threshold,
            section06_chunk_size=section06_chunk_size,
            section09_chunk_threshold=section09_chunk_threshold,
            section09_chunk_size=section09_chunk_size,
        )

        month_compare_df, lake_agg_df, excel_meta = build_monthly_compare(
            report_month_label=report_month_label,
            final_df=final_df_month,
            excel_path=excel_path,
            excel_header=excel_header_current,
        )

        final_df_format = 'parquet'
        final_df_checkpoint_path = month_final_parquet_path
        try:
            final_df_month.to_parquet(month_final_parquet_path, index=False)
        except Exception as exc_parquet:
            final_df_format = 'csv_fallback'
            final_df_checkpoint_path = month_final_csv_path
            final_df_month.to_csv(month_final_csv_path, index=False, encoding='utf-8-sig')
            print(f'[{report_month_label}] checkpoint_parquet_fallback_to_csv: {type(exc_parquet).__name__}')

        month_compare_df.to_csv(month_compare_csv_path, index=False, encoding='utf-8-sig')

        elapsed = round(time.perf_counter() - month_start_ts, 2)
        print(
            f'[{report_month_label}] checkpoint_saved: final_df_rows={len(final_df_month):,}, '
            f'ref_status={month_compare_df["reference_status"].iloc[0]}, '
            f'excel_header={excel_header_current}, format={final_df_format}, elapsed_sec={elapsed}'
        )

    if lake_agg_df is None:
        lake_agg_df = build_lake_agg(final_df_month)

    if month_compare_df is None:
        month_compare_df, lake_agg_df, excel_meta = build_monthly_compare(
            report_month_label=report_month_label,
            final_df=final_df_month,
            excel_path=excel_path,
            excel_header=excel_header_current,
        )

    final_df_by_month[report_month_label] = final_df_month
    monthly_compare_parts.append(month_compare_df)
    lake_agg_by_month[report_month_label] = lake_agg_df
    excel_meta_by_month[report_month_label] = excel_meta

    run_state['months'][report_month_label] = {
        'status': 'completed',
        'final_df_path': str(final_df_checkpoint_path) if final_df_checkpoint_path is not None else None,
        'final_df_format': final_df_format,
        'monthly_compare_path': str(month_compare_csv_path),
        'final_df_rows': int(len(final_df_month)) if final_df_month is not None else 0,
        'monthly_compare_rows': int(len(month_compare_df)) if month_compare_df is not None else 0,
        'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    }

    completed_months = set(run_state.get('completed_months', []))
    completed_months.add(report_month_label)
    run_state['completed_months'] = sorted(completed_months)
    _save_run_state(monthly_run_state_path, run_state)

if final_df_by_month:
    final_df_period_df = pd.concat(final_df_by_month.values(), ignore_index=True)
else:
    final_df_period_df = pd.DataFrame()

if monthly_compare_parts:
    monthly_compare_df = pd.concat(monthly_compare_parts, ignore_index=True)
else:
    monthly_compare_df = pd.DataFrame(
        columns=[
            'report_month', 'metric', 'lake_value', 'excel_value',
            'delta_lake_minus_excel', 'divergence_pct', 'reference_status', 'excel_path'
        ]
    )

print(f'final_df_period_df rows: {len(final_df_period_df):,}')
print(f'monthly_compare_df rows: {len(monthly_compare_df):,}')
print(f'run_state saved to: {monthly_run_state_path}')
display(monthly_compare_df.head(18))


In [ ]:
# Kedr Общий ЧОД helper — join from lake snapshot (not live Kedr.v_detail_*)
def enrich_final_df_with_kedr_obshiy_chod(
    final_df,
    imp_conn,
    *,
    chunk_size=800,
    mem_limit='8g',
    overlap_csv_path=None,
    rewrite_csv_path=None,
    kedr_table=None,
):
    """Attach kedr_obshiy_chod / kedr_obshiy_chod_contrib from lake table inn x yearmm."""
    if final_df is None or len(final_df) == 0:
        raise RuntimeError('final_df empty — cannot enrich Kedr')

    table = kedr_table or globals().get(
        'kedr_obshiy_chod_table',
        'sandbox_ai.shestopalov_kedr_obshiy_chod_inn_month',
    )

    def _report_month_to_yearmm(v):
        if pd.isna(v):
            return None
        s = str(v).strip()
        if not s:
            return None
        s7 = s[:7]
        if len(s7) >= 7 and s7[4] == '-':
            return int(s7[:4] + s7[5:7])
        digits = re.sub(r'\D+', '', s)
        if len(digits) >= 6:
            return int(digits[:6])
        return None

    out = final_df.copy()
    base = out.copy()
    base['inn_key'] = base['inn'].map(normalize_inn_q1)
    base['yearmm'] = base['report_month'].map(_report_month_to_yearmm)

    yearmms = sorted({int(x) for x in base['yearmm'].dropna().tolist()})
    if not yearmms:
        raise RuntimeError('No yearmm derived from report_month — cannot join Kedr lake table')

    yearmm_sql = ', '.join(str(x) for x in yearmms)
    sql_lake = f"""
    select
      cast(inn as string) as inn,
      cast(yearmm as bigint) as yearmm,
      cast(kedr_obshiy_chod as double) as kedr_obshiy_chod,
      cast(src_cnt as bigint) as src_cnt
    from {table}
    where cast(yearmm as bigint) in ({yearmm_sql})
    """
    print(f'[kedr] reading lake table {table} yearmm in ({yearmm_sql})')
    try:
        with imp_conn:
            imp_conn.execute(f'set MEM_LIMIT={mem_limit}')
            kedr_chod_df = imp_conn.fetch(sql_lake)
    except Exception as exc:
        raise RuntimeError(
            f'Cannot read {table}. Сначала прогони 01_07_build_kedr_obshiy_chod_drp.ipynb. '
            f'{type(exc).__name__}: {exc}'
        ) from exc

    if kedr_chod_df is None or len(kedr_chod_df) == 0:
        print(f'[kedr] WARN: lake table empty for yearmm={yearmms}')
        kedr_chod_df = pd.DataFrame(columns=['inn', 'yearmm', 'kedr_obshiy_chod', 'src_cnt'])
    else:
        kedr_chod_df = kedr_chod_df.copy()
        kedr_chod_df.columns = [str(c).strip().lower() for c in kedr_chod_df.columns]
        kedr_chod_df['inn_key'] = kedr_chod_df['inn'].map(normalize_inn_q1)
        kedr_chod_df['yearmm'] = pd.to_numeric(kedr_chod_df['yearmm'], errors='coerce').astype('Int64')
        kedr_chod_df = (
            kedr_chod_df.dropna(subset=['inn_key', 'yearmm'])
            .groupby(['inn_key', 'yearmm'], as_index=False)
            .agg(
                kedr_obshiy_chod=('kedr_obshiy_chod', 'max'),
                src_cnt=('src_cnt', 'max'),
            )
        )

    print(f'[kedr] lake rows after agg={len(kedr_chod_df):,}')

    for c in ['kedr_obshiy_chod', 'kedr_obshiy_chod_contrib', 'kedr_src_cnt']:
        if c in out.columns:
            out = out.drop(columns=[c])

    merge_df = base[['inn_key', 'yearmm']].merge(
        kedr_chod_df[['inn_key', 'yearmm', 'kedr_obshiy_chod', 'src_cnt']],
        on=['inn_key', 'yearmm'],
        how='left',
    )
    out['kedr_obshiy_chod'] = pd.to_numeric(merge_df['kedr_obshiy_chod'], errors='coerce')
    out['kedr_src_cnt'] = pd.to_numeric(merge_df['src_cnt'], errors='coerce')

    tmp = out.copy()
    tmp['_inn_key'] = tmp['inn'].map(normalize_inn_q1)
    tmp['_rm'] = tmp['report_month'].astype(str).str.strip().str[:7]
    tmp['_agr'] = tmp['agr_id'].astype(str).str.strip()
    tmp['_rn'] = tmp.groupby(['_inn_key', '_rm'], dropna=False)['_agr'].rank(
        method='first', ascending=True
    )
    out['kedr_obshiy_chod_contrib'] = (
        pd.to_numeric(out['kedr_obshiy_chod'], errors='coerce')
        .where(tmp['_rn'] == 1, 0.0)
        .fillna(0.0)
    )

    filled_pct = float(out['kedr_obshiy_chod'].notna().mean() * 100)
    uniq = out.copy()
    uniq['_inn'] = uniq['inn'].map(normalize_inn_q1)
    uniq['_rm'] = uniq['report_month'].astype(str).str.strip().str[:7]
    uniq = uniq.loc[uniq['kedr_obshiy_chod'].notna()].drop_duplicates(subset=['_inn', '_rm'])
    sum_uniq = float(pd.to_numeric(uniq['kedr_obshiy_chod'], errors='coerce').fillna(0).sum())
    sum_contrib = float(pd.to_numeric(out['kedr_obshiy_chod_contrib'], errors='coerce').fillna(0).sum())
    print('[kedr] QC filled={:.2f}% | sum_uniq={:,.2f} | sum_contrib={:,.2f} | drift={:,.6f}'.format(
        filled_pct, sum_uniq, sum_contrib, abs(sum_uniq - sum_contrib)
    ))

    if overlap_csv_path is not None:
        try:
            ov = out.loc[pd.to_numeric(out['kedr_src_cnt'], errors='coerce').fillna(0) > 1, [
                c for c in ['report_month', 'inn', 'kedr_obshiy_chod', 'kedr_src_cnt'] if c in out.columns
            ]].drop_duplicates()
            if len(ov):
                ov.to_csv(overlap_csv_path, index=False, encoding='utf-8-sig')
                print('[kedr] overlap csv:', overlap_csv_path)
        except Exception as exc:
            print('[kedr] WARN overlap csv:', type(exc).__name__)

    if rewrite_csv_path is not None:
        try:
            out.to_csv(rewrite_csv_path, index=False, encoding='utf-8-sig')
            print('[kedr] saved:', rewrite_csv_path)
        except Exception as exc:
            print('[kedr] WARN save csv:', type(exc).__name__)

    return out


print('Kedr helper ready: enrich_final_df_with_kedr_obshiy_chod() ← lake table')


In [ ]:
# Формируем period_total и reference_coverage (Jan–Jul)
expected_months = [d.strftime('%Y-%m') for d in period_months]

# --- rebuild monthly_compare if stale (missing aur / fin_result / …) ---
def _month_metrics_ok(cmp_df, month_label):
    if cmp_df is None or cmp_df.empty:
        return False
    found = set(
        cmp_df.loc[cmp_df['report_month'].astype(str) == month_label, 'metric']
        .astype(str)
        .tolist()
    )
    return set(METRICS_ORDER).issubset(found)


_need_compare_rebuild = (
    'monthly_compare_df' not in globals()
    or monthly_compare_df is None
    or monthly_compare_df.empty
    or any(not _month_metrics_ok(monthly_compare_df, m) for m in expected_months)
)

if _need_compare_rebuild:
    # Prefer in-memory monthly dict; else split period df; else load checkpoints
    _by_month = {}
    if 'final_df_by_month' in globals() and final_df_by_month:
        _by_month = final_df_by_month
    elif 'final_df_period_df' in globals() and final_df_period_df is not None and not final_df_period_df.empty:
        _tmp = final_df_period_df.copy()
        _tmp['report_month'] = _tmp['report_month'].astype(str).str.strip().str[:7]
        _by_month = {m: g.copy() for m, g in _tmp.groupby('report_month')}
    else:
        for month_label in expected_months:
            month_key = month_label.replace('-', '_')
            pq = monthly_checkpoint_dir / f'final_df_{month_key}.parquet'
            csv = monthly_checkpoint_dir / f'final_df_{month_key}.csv'
            if pq.exists():
                _by_month[month_label] = pd.read_parquet(pq)
            elif csv.exists():
                _by_month[month_label] = pd.read_csv(csv)
    if not _by_month:
        raise RuntimeError(
            'monthly_compare stale/missing and no final_df source — '
            're-run helpers (METRICS_ORDER) + month loop cell first'
        )
    print('Rebuilding monthly_compare_df for all months (stale checkpoints / new METRICS_ORDER)...')
    _parts = []
    for month_label in expected_months:
        if month_label not in _by_month or _by_month[month_label] is None:
            raise RuntimeError(f'final_df missing for {month_label}')
        excel_path = excel_reference_by_month.get(month_label)
        excel_header_current = int(excel_header_by_month.get(month_label, excel_header))
        month_key = month_label.replace('-', '_')
        month_compare_csv_path = monthly_checkpoint_dir / f'monthly_compare_{month_key}.csv'
        month_cmp, _, _ = build_monthly_compare(
            report_month_label=month_label,
            final_df=_by_month[month_label],
            excel_path=excel_path,
            excel_header=excel_header_current,
        )
        month_cmp.to_csv(month_compare_csv_path, index=False, encoding='utf-8-sig')
        _parts.append(month_cmp)
        print(
            f'  [{month_label}] compare rebuilt, metrics='
            + str(sorted(month_cmp['metric'].astype(str).unique().tolist()))
        )
    monthly_compare_df = pd.concat(_parts, ignore_index=True)
    print(f'monthly_compare_df rebuilt: rows={len(monthly_compare_df):,}')

reference_coverage_rows = []
for month_label in expected_months:
    excel_path = excel_reference_by_month.get(month_label)
    has_ref = excel_path is not None
    reference_coverage_rows.append({
        'report_month': month_label,
        'has_excel_reference': has_ref,
        'reference_status': 'has_excel_reference' if has_ref else 'no_excel_reference',
        'excel_path': excel_path,
    })

reference_coverage_df = pd.DataFrame(reference_coverage_rows)

with_ref_compare = (
    monthly_compare_df[monthly_compare_df['reference_status'] == 'has_excel_reference']
    .groupby('metric', as_index=False)
    .agg({'lake_value': 'sum', 'excel_value': 'sum'})
)
with_ref_compare['delta_lake_minus_excel'] = with_ref_compare['lake_value'] - with_ref_compare['excel_value']
with_ref_compare['divergence_pct'] = with_ref_compare.apply(
    lambda r: safe_divergence_pct(r['delta_lake_minus_excel'], r['excel_value']), axis=1
)
with_ref_compare['scope'] = 'months_with_excel_reference'
with_ref_compare['reference_status'] = 'has_excel_reference'

no_ref_lake_only = (
    monthly_compare_df[monthly_compare_df['reference_status'] == 'no_excel_reference']
    .groupby('metric', as_index=False)
    .agg({'lake_value': 'sum'})
)
no_ref_lake_only['excel_value'] = np.nan
no_ref_lake_only['delta_lake_minus_excel'] = np.nan
no_ref_lake_only['divergence_pct'] = np.nan
no_ref_lake_only['scope'] = 'months_without_excel_reference'
no_ref_lake_only['reference_status'] = 'no_excel_reference'

period_total_df = pd.concat([with_ref_compare, no_ref_lake_only], ignore_index=True)
period_total_df = period_total_df[
    ['scope', 'metric', 'lake_value', 'excel_value', 'delta_lake_minus_excel', 'divergence_pct', 'reference_status']
]

if final_df_period_df.empty:
    raise RuntimeError('QC fail: final_df_period_df пустой')

final_months_present = sorted(final_df_period_df['report_month'].dropna().astype(str).unique().tolist())
if final_months_present != expected_months:
    raise RuntimeError(
        f'QC fail: months in final_df_period_df={final_months_present}, expected={expected_months}'
    )

months_with_ref = [m for m in expected_months if m in excel_reference_by_month]
months_without_ref = [m for m in expected_months if m not in excel_reference_by_month]

with_ref_cmp = monthly_compare_df[monthly_compare_df['report_month'].isin(months_with_ref)]
for m in months_with_ref:
    metrics_found = sorted(with_ref_cmp[with_ref_cmp['report_month'] == m]['metric'].tolist())
    if sorted(metrics_found) != sorted(METRICS_ORDER):
        raise RuntimeError(f'QC fail: month {m} has incomplete metric comparison: {metrics_found}')
    status = with_ref_cmp[with_ref_cmp['report_month'] == m]['reference_status']
    if status.empty or status.iloc[0] != 'has_excel_reference':
        raise RuntimeError(f'QC fail: month {m} must have Excel reference')

no_ref_cmp = monthly_compare_df[monthly_compare_df['report_month'].isin(months_without_ref)]
if not no_ref_cmp.empty:
    if (
        no_ref_cmp['reference_status'].nunique() != 1
        or no_ref_cmp['reference_status'].iloc[0] != 'no_excel_reference'
    ):
        raise RuntimeError(
            f'QC fail: months without Excel ref must be no_excel_reference: {months_without_ref}'
        )

prod_missing = [
    c for c in PRODUCT_FLAG_COLUMNS + ['recommended_products']
    if c not in final_df_period_df.columns
]
if prod_missing:
    raise RuntimeError(f'QC fail: missing product columns in final_df: {prod_missing}')

output_dir.mkdir(parents=True, exist_ok=True)
final_df_period_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

with pd.ExcelWriter(output_compare_excel_path, engine='openpyxl') as writer:
    monthly_compare_df.to_excel(writer, sheet_name='monthly_compare', index=False)
    period_total_df.to_excel(writer, sheet_name='period_total', index=False)
    reference_coverage_df.to_excel(writer, sheet_name='reference_coverage', index=False)

period_unique_inn = final_df_period_df['inn'].apply(normalize_inn_q1).dropna().nunique()

print('Saved files:')
print(f'  CSV:   {output_csv_path}')
print(f'  Excel: {output_compare_excel_path}')
print('Control stats:')
print(f'  final_df_period_df rows: {len(final_df_period_df):,}')
print(f'  final_df_period_df unique INN: {period_unique_inn:,}')
print(f'  months in final_df_period_df: {final_months_present}')
print(f'  months with Excel: {months_with_ref}')
print(f'  months without Excel: {months_without_ref}')
print(f'  product flag cols: OK ({len(PRODUCT_FLAG_COLUMNS)} + recommended_products)')

display(reference_coverage_df)
display(period_total_df)


# --- Smoke: amortization / chod / fin_result after rebuild ---
_smoke_cols = [c for c in ['amortization', 'chod', 'fin_result', 'aur'] if c in final_df_period_df.columns]
if _smoke_cols:
    smoke_by_month = (
        final_df_period_df.assign(
            **{c: pd.to_numeric(final_df_period_df[c], errors='coerce').fillna(0.0) for c in _smoke_cols}
        )
        .groupby('report_month', as_index=False)[_smoke_cols]
        .sum()
        .sort_values('report_month')
    )
    print('=== Smoke period totals (lake final_df) ===')
    display(smoke_by_month)
    for c in _smoke_cols:
        print(f'  total {c} =', float(smoke_by_month[c].sum()))
else:
    print('Smoke skip: amortization/chod/fin_result columns missing')


# --- Auto: Kedr Общий ЧОД (inn x month) before QC / DRP ---
if run_kedr_obshiy_chod_enrich:
    print('=== Auto Kedr enrich ===')
    final_df_period_df = enrich_final_df_with_kedr_obshiy_chod(
        final_df_period_df,
        imp,
        chunk_size=kedr_chunk_size,
        mem_limit=kedr_mem_limit,
        overlap_csv_path=kedr_overlap_csv_path,
        rewrite_csv_path=output_csv_path,
        kedr_table=kedr_obshiy_chod_table,
    )
else:
    print('SKIP auto Kedr enrich (run_kedr_obshiy_chod_enrich=False)')


# QC: active_retl_cnt <= retl_cnt (local dataframe, before/after Kedr enrich)
if 'final_df_period_df' in globals() and final_df_period_df is not None and len(final_df_period_df):
    _qc = final_df_period_df.copy()
    if 'active_retl_cnt' not in _qc.columns:
        print('QC active_retl: column missing — recompute months with updated section 05')
    else:
        _arc = pd.to_numeric(_qc['active_retl_cnt'], errors='coerce').fillna(0)
        _rc = pd.to_numeric(_qc['retl_cnt'], errors='coerce').fillna(0)
        _bad = int((_arc > _rc).sum())
        print(
            'QC active_retl: sum_retl=%s sum_active_retl=%s rows_active_gt_retl=%s sum_active_terms=%s'
            % (
                int(_rc.sum()),
                int(_arc.sum()),
                _bad,
                int(pd.to_numeric(_qc.get('active_terms'), errors='coerce').fillna(0).sum()),
            )
        )
        if _bad:
            raise AssertionError('active_retl_cnt > retl_cnt on %s rows' % _bad)
else:
    print('QC active_retl: skipped (no final_df_period_df)')


## QC: амортизация по месяцам — lake vs Excel

Сверка после нового источника `sandbox_ai.shestopalov_terminal_amortization_model_jan_aug`.

Берёт `monthly_compare_df` (metric=`amortization`): lake vs Excel `Амортизация`, delta и %.
Не зависит от `run_excel_qc` — месяцы без Excel будут с `no_excel_reference`.


In [ ]:
# Amortization by month: lake (new model) vs Excel
if 'monthly_compare_df' not in globals() or monthly_compare_df is None or monthly_compare_df.empty:
    raise RuntimeError('monthly_compare_df missing — сначала period cell / month loop')

amort_cmp = monthly_compare_df.loc[
    monthly_compare_df['metric'].astype(str) == 'amortization'
].copy()

if amort_cmp.empty:
    raise RuntimeError('No amortization rows in monthly_compare_df — check METRICS_ORDER / rebuild compare')

amort_by_month = amort_cmp[[
    'report_month',
    'lake_value',
    'excel_value',
    'delta_lake_minus_excel',
    'divergence_pct',
    'reference_status',
]].copy()

amort_by_month = amort_by_month.rename(columns={
    'lake_value': 'amortization_lake',
    'excel_value': 'amortization_excel',
    'delta_lake_minus_excel': 'delta_lake_minus_excel',
    'divergence_pct': 'delta_pct_vs_excel',
})

for c in ['amortization_lake', 'amortization_excel', 'delta_lake_minus_excel', 'delta_pct_vs_excel']:
    amort_by_month[c] = pd.to_numeric(amort_by_month[c], errors='coerce')

amort_by_month = amort_by_month.sort_values('report_month').reset_index(drop=True)

# Period totals (only months with Excel reference for fair compare)
with_xl = amort_by_month.loc[amort_by_month['reference_status'] == 'has_excel_reference']
sum_lake_all = float(amort_by_month['amortization_lake'].fillna(0).sum())
sum_lake_xl = float(with_xl['amortization_lake'].fillna(0).sum()) if len(with_xl) else 0.0
sum_excel = float(with_xl['amortization_excel'].fillna(0).sum()) if len(with_xl) else 0.0
sum_delta = sum_lake_xl - sum_excel
sum_delta_pct = (sum_delta / sum_excel * 100.0) if abs(sum_excel) > 1e-9 else np.nan

print('=== Amortization lake vs Excel by month ===')
print('(lake = sandbox_ai.shestopalov_terminal_amortization_model_jan_aug → final_df.amortization)')
display(amort_by_month)

print('=== Period totals (months with Excel reference) ===')
print(f'  amortization_lake  = {sum_lake_xl:,.2f}')
print(f'  amortization_excel = {sum_excel:,.2f}')
print(f'  delta (lake-excel) = {sum_delta:,.2f}')
print(f'  delta_pct          = {sum_delta_pct:,.2f}%' if pd.notna(sum_delta_pct) else '  delta_pct = n/a')
print(f'  lake all months (incl. no Excel) = {sum_lake_all:,.2f}')

if pd.notna(sum_delta_pct):
    if abs(sum_delta_pct) < 5:
        print('Verdict: lake close to Excel (|delta| < 5% on months with reference).')
    elif sum_delta > 0:
        print('Verdict: lake amortization HIGHER than Excel on referenced months.')
    else:
        print('Verdict: lake amortization LOWER than Excel on referenced months.')

amort_qc_csv = output_dir / 'amortization_lake_vs_excel_by_month_2026_01_2026_07.csv'
amort_by_month.to_csv(amort_qc_csv, index=False, encoding='utf-8-sig')
print('Saved:', amort_qc_csv)


## Kedr «Общий ЧОД» (из озера)

Источник: `sandbox_ai.shestopalov_kedr_obshiy_chod_inn_month`  
(сборка: `01_07_build_kedr_obshiy_chod_drp.ipynb` — preflight месяцев + batch из Kedr).

Автовызов в конце period-ячейки. Ниже — ручной повтор join из озера.


In [ ]:
# Optional re-run Kedr enrich (already called automatically after period)
if 'final_df_period_df' not in globals() or final_df_period_df is None or final_df_period_df.empty:
    raise RuntimeError('Сначала соберите final_df_period_df (месячный цикл + period cell).')

if run_kedr_obshiy_chod_enrich:
    final_df_period_df = enrich_final_df_with_kedr_obshiy_chod(
        final_df_period_df,
        imp,
        chunk_size=kedr_chunk_size,
        mem_limit=kedr_mem_limit,
        overlap_csv_path=kedr_overlap_csv_path,
        rewrite_csv_path=output_csv_path,
        kedr_table=kedr_obshiy_chod_table,
    )
    display(
        final_df_period_df[
            [c for c in [
                'report_month', 'inn', 'agr_id', 'chod',
                'kedr_obshiy_chod', 'kedr_obshiy_chod_contrib', 'kedr_src_cnt',
            ] if c in final_df_period_df.columns]
        ].head(15)
    )
else:
    print('SKIP Kedr enrich: run_kedr_obshiy_chod_enrich=False')


## QC — TOP-10 Excel vs озеро по commission_monthly

Для каждого месяца с Excel:
- join lake ↔ Excel по `inn_key + agr_id_key`
- TOP-10 по `|lake - excel|` для `commission_monthly`
- список `c_nmrc` (agr_terms ∩ SA agreements) за месяц

Май без Excel — skip.


In [ ]:
if not run_excel_qc:
    print('SKIP Excel QC cell (run_excel_qc=False)')
else:
    # === QC: TOP-10 inn+agr_id by |delta| commission_monthly Excel vs lake + c_nmrc list ===
    exact_abs_tol = 0.01
    top_n = 10
    
    if 'final_df_by_month' not in globals() or not final_df_by_month:
        raise RuntimeError('final_df_by_month is empty — run the month loop first')
    
    if 'imp' not in globals() or imp is None:
        raise RuntimeError('Impala connection `imp` is required for c_nmrc lookup')
    
    
    def _month_bounds(month_label):
        start = pd.Timestamp(f'{month_label}-01')
        end = (start + pd.offsets.MonthEnd(0)).normalize()
        return start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')
    
    
    def _fetch_cnmrc_for_agrs(agr_ids, month_start, month_end):
        agr_ids = [str(x) for x in agr_ids if x]
        if not agr_ids:
            return pd.DataFrame(columns=['agr_id_key', 'c_nmrc_list', 'c_nmrc_cnt'])
        ids_sql = ','.join("'" + x.replace("'", "") + "'" for x in agr_ids)
        sql = f"""
    with terms_active as (
      select distinct
        cast(t.n_agr as string) as n_agr,
        cast(t.c_nmrc as string) as c_nmrc
      from ods_alpha.scd1_agr_terms t
      where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
        and t.c_nmrc is not null
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
    ),
    agreements_sa as (
      select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as agr_id
      from ods_alpha.scd1_agreements a
      where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
        and upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.abs_agr_id as string) in ({ids_sql})
        and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
    )
    select distinct
      a.agr_id as agr_id_key,
      t.c_nmrc
    from agreements_sa a
    left join terms_active t on t.n_agr = a.n_agr
    """
        with imp:
            imp.execute('set MEM_LIMIT=4g')
            df = imp.fetch(sql)
        if df is None or df.empty:
            return pd.DataFrame(columns=['agr_id_key', 'c_nmrc_list', 'c_nmrc_cnt'])
        tmp = df.copy()
        tmp['agr_id_key'] = tmp['agr_id_key'].map(normalize_agr_q1)
        tmp['c_nmrc'] = tmp['c_nmrc'].astype(str).where(tmp['c_nmrc'].notna(), None)
        rows = []
        for agr, g in tmp.groupby('agr_id_key', dropna=False):
            vals = sorted({str(v) for v in g['c_nmrc'].dropna().tolist() if str(v) not in {'', 'None', 'nan'}})
            rows.append({
                'agr_id_key': agr,
                'c_nmrc_list': ','.join(vals),
                'c_nmrc_cnt': len(vals),
            })
        return pd.DataFrame(rows)
    
    
    top10_parts = []
    expected_months = [d.strftime('%Y-%m') for d in period_months]
    
    for month_label in expected_months:
        excel_path = excel_reference_by_month.get(month_label)
        print(f'\n=== {month_label} ===')
        if excel_path is None:
            print('no_excel_reference — skip TOP-10')
            continue
        if month_label not in final_df_by_month or final_df_by_month[month_label] is None:
            print('final_df missing — skip')
            continue
    
        excel_path = Path(excel_path)
        if not excel_path.exists():
            print(f'Excel file not found: {excel_path} — skip')
            continue
    
        fd = final_df_by_month[month_label].copy()
        fd['inn_key'] = fd['inn'].map(normalize_inn_q1)
        fd['agr_id_key'] = fd['agr_id'].map(normalize_agr_q1)
        fd['commission_monthly_lake'] = pd.to_numeric(fd.get('commission_monthly'), errors='coerce').fillna(0.0)
        lake_agg = (
            fd.dropna(subset=['inn_key', 'agr_id_key'])
            .groupby(['inn_key', 'agr_id_key'], as_index=False)
            .agg(commission_monthly_lake=('commission_monthly_lake', 'sum'))
        )
    
        excel_header_current = int(excel_header_by_month.get(month_label, excel_header))
        ex_agg, resolved = load_excel_agg(str(excel_path), excel_header=excel_header_current)
        ex_agg = ex_agg.copy()
        ex_agg['commission_monthly_excel'] = pd.to_numeric(
            ex_agg['commission_monthly_excel'], errors='coerce'
        ).fillna(0.0)
    
        cmp = lake_agg.merge(
            ex_agg[['inn_key', 'agr_id_key', 'commission_monthly_excel']],
            on=['inn_key', 'agr_id_key'],
            how='outer',
        )
        cmp['commission_monthly_lake'] = cmp['commission_monthly_lake'].fillna(0.0)
        cmp['commission_monthly_excel'] = cmp['commission_monthly_excel'].fillna(0.0)
        cmp['delta_lake_minus_excel'] = cmp['commission_monthly_lake'] - cmp['commission_monthly_excel']
        cmp['abs_delta'] = cmp['delta_lake_minus_excel'].abs()
    
        top = (
            cmp.loc[cmp['abs_delta'] > exact_abs_tol]
            .sort_values('abs_delta', ascending=False)
            .head(top_n)
            .copy()
        )
        if top.empty:
            print('No deltas above tolerance')
            continue
    
        month_start, month_end = _month_bounds(month_label)
        cnmrc = _fetch_cnmrc_for_agrs(top['agr_id_key'].tolist(), month_start, month_end)
        top = top.merge(cnmrc, on='agr_id_key', how='left')
        top['c_nmrc_cnt'] = top['c_nmrc_cnt'].fillna(0).astype(int)
        top['c_nmrc_list'] = top['c_nmrc_list'].fillna('')
        top.insert(0, 'report_month', month_label)
        top.insert(1, 'rank', range(1, len(top) + 1))
    
        show_cols = [
            'report_month', 'rank', 'inn_key', 'agr_id_key',
            'commission_monthly_excel', 'commission_monthly_lake',
            'delta_lake_minus_excel', 'abs_delta', 'c_nmrc_cnt', 'c_nmrc_list',
        ]
        top_show = top[show_cols].rename(columns={'inn_key': 'inn', 'agr_id_key': 'agr_id'})
        display(top_show)
        top10_parts.append(top_show)
        print(
            f'TOP-{len(top)} | sum |delta|={top["abs_delta"].sum():,.2f} | '
            f'sum excel={top["commission_monthly_excel"].sum():,.2f} | '
            f'sum lake={top["commission_monthly_lake"].sum():,.2f}'
        )
    
    if top10_parts:
        top10_delta_by_month_df = pd.concat(top10_parts, ignore_index=True)
    else:
        top10_delta_by_month_df = pd.DataFrame(
            columns=[
                'report_month', 'rank', 'inn', 'agr_id',
                'commission_monthly_excel', 'commission_monthly_lake',
                'delta_lake_minus_excel', 'abs_delta', 'c_nmrc_cnt', 'c_nmrc_list',
            ]
        )
    
    output_dir.mkdir(parents=True, exist_ok=True)
    top10_delta_by_month_df.to_csv(output_top10_csv_path, index=False, encoding='utf-8-sig')
    
    # Append / replace sheet in compare workbook if it exists
    if output_compare_excel_path.exists():
        with pd.ExcelWriter(
            output_compare_excel_path,
            engine='openpyxl',
            mode='a',
            if_sheet_exists='replace',
        ) as writer:
            top10_delta_by_month_df.to_excel(writer, sheet_name='top10_delta_by_month', index=False)
    else:
        with pd.ExcelWriter(output_compare_excel_path, engine='openpyxl') as writer:
            top10_delta_by_month_df.to_excel(writer, sheet_name='top10_delta_by_month', index=False)
    
    print('\nSaved TOP-10:')
    print(f'  CSV:   {output_top10_csv_path}')
    print(f'  Excel sheet top10_delta_by_month in {output_compare_excel_path}')
    print(f'  rows: {len(top10_delta_by_month_df):,}')
    display(top10_delta_by_month_df)


## QC — точное совпадение строк Excel vs озеро

Зерно: `inn_key + agr_id_key` (`normalize_inn_q1`).

Метрики (default): `commission_monthly`, `trx_cnt`, `trx_sum`, `retl_cnt`, `term_cnt`.

Для каждого месяца с Excel:
- outer join lake ↔ excel
- % exact match на ключах `both` (допуск 0.01 для денег/сумм; 0 для счётчиков)
- counts: only_excel / only_lake / both


In [ ]:
if not run_excel_qc:
    print('SKIP Excel QC cell (run_excel_qc=False)')
else:
    # === QC: row exact match Excel vs lake (inn+agr_id) ===
    exact_abs_tol_money = 0.01
    ROW_MATCH_METRICS = [
        # metric, lake_col, excel_col, tol
        ('retl_cnt', 'retl_cnt_lake', 'retl_cnt_excel', 0.0),
        ('term_cnt', 'term_cnt_lake', 'term_cnt_excel', 0.0),
        ('trx_cnt', 'trx_cnt_lake', 'trx_cnt_excel', 0.0),
        ('trx_sum', 'trx_sum_lake', 'trx_sum_excel', exact_abs_tol_money),
        ('commission_monthly', 'commission_monthly_lake', 'commission_monthly_excel', exact_abs_tol_money),
        ('aur', 'aur_lake', 'aur_excel', exact_abs_tol_money),
        ('amortization', 'amortization_lake', 'amortization_excel', exact_abs_tol_money),
        ('fin_result', 'fin_result_lake', 'fin_result_excel', exact_abs_tol_money),
    ]
    
    if 'final_df_by_month' not in globals() or not final_df_by_month:
        raise RuntimeError('final_df_by_month is empty — run the month loop first')
    
    row_match_parts = []
    expected_months = [d.strftime('%Y-%m') for d in period_months]
    
    for month_label in expected_months:
        excel_path = excel_reference_by_month.get(month_label)
        print(f'\n=== row match {month_label} ===')
        if excel_path is None:
            print('no_excel_reference — skip')
            continue
        if month_label not in final_df_by_month or final_df_by_month[month_label] is None:
            print('final_df missing — skip')
            continue
        excel_path = Path(excel_path)
        if not excel_path.exists():
            print(f'Excel not found: {excel_path} — skip')
            continue
    
        lake_agg = build_lake_agg(final_df_by_month[month_label])
        excel_header_current = int(excel_header_by_month.get(month_label, excel_header))
        ex_agg, _ = load_excel_agg(str(excel_path), excel_header=excel_header_current)
    
        cmp = lake_agg.merge(ex_agg, on=['inn_key', 'agr_id_key'], how='outer', indicator=True)
        n_both = int((cmp['_merge'] == 'both').sum())
        n_only_excel = int((cmp['_merge'] == 'right_only').sum())
        n_only_lake = int((cmp['_merge'] == 'left_only').sum())
        print(f'keys: both={n_both:,}, only_excel={n_only_excel:,}, only_lake={n_only_lake:,}')
    
        both = cmp.loc[cmp['_merge'] == 'both'].copy()
        for metric, lake_col, excel_col, tol in ROW_MATCH_METRICS:
            if lake_col not in both.columns or excel_col not in both.columns:
                continue
            # skip optional Excel columns that were not found (all-NaN)
            if pd.to_numeric(both[excel_col], errors='coerce').notna().sum() == 0:
                print(f'  {metric}: skip (no Excel column / all empty)')
                continue
            lv = pd.to_numeric(both[lake_col], errors='coerce').fillna(0.0)
            ev = pd.to_numeric(both[excel_col], errors='coerce').fillna(0.0)
            if tol and tol > 0:
                exact = np.isclose(lv, ev, atol=tol, rtol=0)
            else:
                exact = lv.eq(ev)
            exact_n = int(exact.sum())
            exact_pct = round(100.0 * exact_n / n_both, 2) if n_both else None
            row_match_parts.append({
                'report_month': month_label,
                'metric': metric,
                'keys_both': n_both,
                'keys_only_excel': n_only_excel,
                'keys_only_lake': n_only_lake,
                'exact_match_n': exact_n,
                'exact_match_pct': exact_pct,
                'mismatch_n': n_both - exact_n,
                'tol': tol,
            })
            print(f'  {metric}: exact={exact_pct}% ({exact_n:,}/{n_both:,}), mismatch={n_both - exact_n:,}')
    
    if row_match_parts:
        row_exact_match_df = pd.DataFrame(row_match_parts)
    else:
        row_exact_match_df = pd.DataFrame(
            columns=[
                'report_month', 'metric', 'keys_both', 'keys_only_excel', 'keys_only_lake',
                'exact_match_n', 'exact_match_pct', 'mismatch_n', 'tol',
            ]
        )
    
    output_dir.mkdir(parents=True, exist_ok=True)
    _row_path = globals().get('output_row_match_csv_path', output_dir / 'row_exact_match_2026_01_2026_07_mpos.csv')
    row_exact_match_df.to_csv(_row_path, index=False, encoding='utf-8-sig')
    
    if output_compare_excel_path.exists():
        with pd.ExcelWriter(
            output_compare_excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace'
        ) as writer:
            row_exact_match_df.to_excel(writer, sheet_name='row_exact_match', index=False)
    else:
        with pd.ExcelWriter(output_compare_excel_path, engine='openpyxl') as writer:
            row_exact_match_df.to_excel(writer, sheet_name='row_exact_match', index=False)
    
    print(f'\nSaved row exact match: {_row_path}')
    display(row_exact_match_df)


## Аналитическая записка Excel vs final_df

Собирает markdown-записку из:
1. абсолютных дельт (`monthly_compare_df`)
2. % точного совпадения строк (`row_exact_match_df`)

Файл: `memo_excel_vs_final_df_2026_01_2026_07_mpos.md`


In [ ]:
if not run_excel_qc:
    print('SKIP Excel QC cell (run_excel_qc=False)')
else:
    # === ANALYTICAL MEMO: Excel vs final_df ===
    from io import StringIO
    
    if 'monthly_compare_df' not in globals() or monthly_compare_df is None or monthly_compare_df.empty:
        raise RuntimeError('monthly_compare_df missing — run month loop + period_total cell first')
    if 'row_exact_match_df' not in globals() or row_exact_match_df is None:
        raise RuntimeError('row_exact_match_df missing — run row exact match QC cell first')
    
    memo_path = globals().get(
        'output_memo_md_path',
        output_dir / 'memo_excel_vs_final_df_2026_01_2026_07_mpos.md',
    )
    
    cmp = monthly_compare_df[
        monthly_compare_df['reference_status'] == 'has_excel_reference'
    ].copy()
    for c in ['lake_value', 'excel_value', 'delta_lake_minus_excel', 'divergence_pct']:
        if c in cmp.columns:
            cmp[c] = pd.to_numeric(cmp[c], errors='coerce')
    
    # Count-like metrics for section 1 (absolute quantity gaps)
    COUNT_METRICS = ['unique_inn', 'retl_cnt', 'term_cnt', 'trx_cnt']
    MONEY_METRICS = ['trx_sum', 'commission_from_ops', 'commission_monthly', 'commission_total', 'int_component', 'chod', 'aur', 'amortization', 'fin_result']
    
    abs_count = cmp.loc[cmp['metric'].isin(COUNT_METRICS)].copy()
    abs_money = cmp.loc[cmp['metric'].isin(MONEY_METRICS)].copy()
    
    # Pivot helpers for readable tables
    def _pivot_delta(df, value_col='delta_lake_minus_excel'):
        if df.empty:
            return pd.DataFrame()
        p = df.pivot_table(
            index='report_month', columns='metric', values=value_col, aggfunc='first'
        )
        # stable column order
        cols = [m for m in (COUNT_METRICS + MONEY_METRICS) if m in p.columns]
        return p.reindex(columns=cols)
    
    
    def _fmt_num(x, nd=2):
        if pd.isna(x):
            return '—'
        try:
            v = float(x)
        except Exception:
            return str(x)
        if abs(v - round(v)) < 1e-9 and abs(v) >= 1:
            return f'{int(round(v)):,}'.replace(',', ' ')
        return f'{v:,.{nd}f}'.replace(',', ' ')
    
    
    def _df_to_md(df):
        if df is None or df.empty:
            return '_нет данных_\n'
        out = df.copy()
        # format numbers
        for c in out.columns:
            if pd.api.types.is_numeric_dtype(out[c]):
                out[c] = out[c].map(lambda x: _fmt_num(x))
        # index as first col if named
        if out.index.name or not isinstance(out.index, pd.RangeIndex):
            out = out.reset_index()
        cols = list(out.columns)
        lines = []
        lines.append('| ' + ' | '.join(str(c) for c in cols) + ' |')
        lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
        for _, row in out.iterrows():
            lines.append('| ' + ' | '.join(str(row[c]) for c in cols) + ' |')
        return '\n'.join(lines) + '\n'
    
    
    delta_count_pivot = _pivot_delta(abs_count)
    delta_money_pivot = _pivot_delta(abs_money)
    lake_count_pivot = _pivot_delta(abs_count, 'lake_value')
    excel_count_pivot = _pivot_delta(abs_count, 'excel_value')
    
    # Row match pivots
    rm = row_exact_match_df.copy()
    if not rm.empty:
        match_pivot = rm.pivot_table(
            index='report_month', columns='metric', values='exact_match_pct', aggfunc='first'
        )
        metric_order = [m for m, *_ in ROW_MATCH_METRICS if m in match_pivot.columns]
        match_pivot = match_pivot.reindex(columns=metric_order)
        key_cov = (
            rm.groupby('report_month', as_index=False)
            .agg(
                keys_both=('keys_both', 'first'),
                keys_only_excel=('keys_only_excel', 'first'),
                keys_only_lake=('keys_only_lake', 'first'),
            )
            .sort_values('report_month')
        )
    else:
        match_pivot = pd.DataFrame()
        key_cov = pd.DataFrame()
    
    # Short verdicts
    def _verdict_lines():
        lines = []
        if not abs_count.empty:
            # worst relative count metric by mean |div|
            tmp = abs_count.copy()
            tmp['abs_div'] = tmp['divergence_pct'].abs()
            worst = tmp.sort_values('abs_div', ascending=False).head(3)
            for _, r in worst.iterrows():
                lines.append(
                    f"- {r['report_month']} / `{r['metric']}`: lake={_fmt_num(r['lake_value'])}, "
                    f"excel={_fmt_num(r['excel_value'])}, delta(lake−excel)={_fmt_num(r['delta_lake_minus_excel'])} "
                    f"({_fmt_num(r['divergence_pct'])}%)"
                )
        if not rm.empty:
            cm = rm.loc[rm['metric'] == 'commission_monthly']
            if not cm.empty:
                med = cm['exact_match_pct'].median()
                lines.append(
                    f"- `commission_monthly`: медиана exact match по строкам ≈ {_fmt_num(med)}% "
                    f"(на ключах both; tol={exact_abs_tol_money})"
                )
            aur_m = rm.loc[rm['metric'] == 'aur']
            if not aur_m.empty:
                med_a = aur_m['exact_match_pct'].median()
                lines.append(
                    f"- `aur`: медиана exact match по строкам ≈ {_fmt_num(med_a)}% "
                    f"(на ключах both; tol={exact_abs_tol_money})"
                )
            am_m = rm.loc[rm['metric'] == 'amortization']
            if not am_m.empty:
                med_am = am_m['exact_match_pct'].median()
                lines.append(
                    f"- `amortization`: медиана exact match по строкам ≈ {_fmt_num(med_am)}% "
                    f"(на ключах both; tol={exact_abs_tol_money})"
                )
            fr_m = rm.loc[rm['metric'] == 'fin_result']
            if not fr_m.empty:
                med_f = fr_m['exact_match_pct'].median()
                lines.append(
                    f"- `fin_result`: медиана exact match по строкам ≈ {_fmt_num(med_f)}% "
                    f"(на ключах both; tol={exact_abs_tol_money})"
                )
        if not lines:
            lines.append('- недостаточно данных для краткого вывода')
        return lines
    
    
    buf = StringIO()
    buf.write('# Аналитическая записка: Excel vs final_df (Jan–Jul 2026)\n\n')
    buf.write('**Источник озера:** `final_df` (тетрадка `01_07_acq_dash_jan_jun_mpos`), ')
    buf.write('`commission_monthly` = MPOS_RENT.n_amt (unique c_nmrc→inn+agr_id).\n\n')
    buf.write('**Зерно строк:** `inn + agr_id` (нормализация ИНН: `normalize_inn_q1`, в т.ч. ведущий 0).\n\n')
    buf.write('## 1. Краткий вывод\n\n')
    for line in _verdict_lines():
        buf.write(line + '\n')
    buf.write('\n')
    
    buf.write('## 2. Абсолютные расхождения количеств (lake − excel)\n\n')
    buf.write('Метрики: `unique_inn`, `retl_cnt`, `term_cnt`, `trx_cnt`.\n\n')
    buf.write('### 2.1. Delta (lake − excel)\n\n')
    buf.write(_df_to_md(delta_count_pivot))
    buf.write('\n### 2.2. Озеро (lake)\n\n')
    buf.write(_df_to_md(lake_count_pivot))
    buf.write('\n### 2.3. Excel\n\n')
    buf.write(_df_to_md(excel_count_pivot))
    
    buf.write('\n## 3. Абсолютные расхождения денежных метрик (lake − excel)\n\n')
    buf.write(_df_to_md(delta_money_pivot))
    
    buf.write('\n## 4. Процент точного совпадения по строкам (inn+agr_id)\n\n')
    buf.write('Считается только на ключах, присутствующих **и в Excel, и в озере** (`both`).\n\n')
    buf.write('### 4.1. Покрытие ключей\n\n')
    buf.write(_df_to_md(key_cov))
    buf.write('\n### 4.2. Exact match, %\n\n')
    buf.write(_df_to_md(match_pivot))
    buf.write('\nДопуски: счётчики — точное равенство; `trx_sum` / `commission_monthly` — atol=0.01.\n\n')
    
    buf.write('## 5. Комментарий по commission_monthly\n\n')
    buf.write(
        'При расхождениях на совпадающих ключах типичная причина (по диагностике June Debug E): '
        'в озере нет строк `MPOS_RENT` за месяц по терминам договора '
        '(`has_terms_but_no_june_rent`) либо договор не в активных SA agreements — '
        'тогда витрина ставит 0, а Excel показывает комиссию.\n\n'
    )
    
    buf.write('## 6. Приложение\n\n')
    buf.write('- Compare workbook: `final_df_compare_2026_01_2026_07_mpos.xlsx` '
              '(листы `monthly_compare`, `period_total`, `row_exact_match`, `top10_delta_by_month`).\n')
    buf.write('- CSV строк: `row_exact_match_2026_01_2026_07_mpos.csv`.\n')
    buf.write('- TOP-10 по |delta| commission_monthly: `top10_commission_delta_2026_01_2026_07_mpos.csv`.\n')
    
    memo_text = buf.getvalue()
    output_dir.mkdir(parents=True, exist_ok=True)
    Path(memo_path).write_text(memo_text, encoding='utf-8')
    print(f'Saved memo: {memo_path}')
    print('\n---- memo preview ----\n')
    print(memo_text)
    
    # also keep in globals
    analytical_memo_md = memo_text


## DRP upload (для дашборда Superset)

Копия из `01_07_acq_dash.ipynb`.

Target: `sbx_da.tmp_shestopalov_acq_datamart_jan_jun`  
Rollback: `sbx_da.tmp_shestopalov_acq_datamart_q1`

Перед запуском нужен готовый `final_df_period_df` (месячный цикл + period cell).

При **Run All**: выполняется если `run_drp_upload = True` (спросит user/password).
Kedr уже должен быть в `final_df_period_df` после period-ячейки.

После upload автоматически: `GRANT SELECT` на новую таблицу роли Superset (`raisa_superset`).
Если GRANT упадёт — таблица залита, но дашборд снова покажет permission denied; тогда выполните GRANT вручную под владельцем.


In [ ]:
# DRP publish config (new table for Superset)
drp_schema_new = 'sbx_da'
drp_table_old = 'tmp_shestopalov_acq_datamart_q1'
drp_table_new = 'tmp_shestopalov_acq_datamart_jan_jun'

dashboard_required_cols = [
    'inn', 'company_name', 'contract_number',
    'd_valid_from', 'd_valid_to',
    'vsp_name', 'vsp_code',
    'tariff_short',
    'retl_cnt', 'active_retl_cnt', 'term_cnt', 'active_terms',
    'commission_total', 'chod', 'fin_result',
    'kedr_obshiy_chod', 'kedr_obshiy_chod_contrib',
    'snapshot_month_start',
    'rko', 'accounting', 'business_cards', 'credit',
    'dbo', 'deposit', 'insurance', 'loyalty_program', 'nmo', 'nso',
    'pravocard', 'salary_project', 'self_inkass', 'service_package', 'sms_info',
    'recommended_products',
]

superset_time_col = 'snapshot_month_start'
# Superset DB role — GRANT SELECT после каждого recreate таблицы
drp_superset_grant_role = 'raisa_superset'

print('DRP old table:', f"{drp_schema_new}.{drp_table_old}")
print('DRP new table:', f"{drp_schema_new}.{drp_table_new}")
print('Superset time column:', superset_time_col)
print('Superset GRANT role:', drp_superset_grant_role)
print('Required cols for smoke:', dashboard_required_cols)


In [ ]:
if not run_drp_upload:
    print('SKIP DRP upload (run_drp_upload=False). final_df_period_df already has Kedr cols if enrich ran.')
else:
    # Upload final_df_period_df to NEW DRP table + schema compatibility checks
    if 'final_df_period_df' not in globals() or final_df_period_df is None or len(final_df_period_df) == 0:
        raise RuntimeError('Сначала выполните расчет периода: final_df_period_df пустой.')
    
    schema_name = str(drp_schema_new).strip().lower()
    old_table_name = str(drp_table_old).strip().lower()
    new_table_name = str(drp_table_new).strip().lower()
    
    target_old_fq = f'{schema_name}.{old_table_name}'
    target_new_fq = f'{schema_name}.{new_table_name}'
    
    print('Preparing upload...')
    print('old table =', target_old_fq)
    print('new table =', target_new_fq)
    print('source rows =', len(final_df_period_df))
    
    drp_user = input('DRP user: ').strip()
    drp_password = getpass('DRP password: ')
    
    drp_conn = connect(
        to='DRP',
        user_params={
            'user_name': drp_user,
            'password': drp_password,
        }
    )
    
    df_upload = final_df_period_df.copy()
    df_upload.columns = [str(c).strip().lower() for c in df_upload.columns]
    
    sql_old_cols = f"""
    select lower(column_name) as column_name
    from information_schema.columns
    where table_schema = '{schema_name}'
      and table_name = '{old_table_name}'
    order by ordinal_position
    """
    
    with drp_conn:
        old_cols_df = drp_conn.fetch(sql_old_cols)
    
    old_cols = []
    if old_cols_df is not None and len(old_cols_df):
        old_cols = [str(x).strip().lower() for x in old_cols_df['column_name'].dropna().tolist()]
    
    print('old table columns =', len(old_cols))
    
    # Compatibility aliases for existing dashboard fields
    compat_alias_map = {
        'branch_rf': 'filial_rf',
        'tarif_name': 'tariff_name',
        'month': 'snapshot_month_start',
    }
    for target_col, source_col in compat_alias_map.items():
        if target_col in old_cols and target_col not in df_upload.columns and source_col in df_upload.columns:
            df_upload[target_col] = df_upload[source_col]
    
    missing_old_cols_before_fill = [c for c in old_cols if c not in df_upload.columns]
    for c in missing_old_cols_before_fill:
        df_upload[c] = None
    
    if old_cols:
        extra_new_cols = [c for c in df_upload.columns if c not in old_cols]
        df_upload = df_upload[old_cols + extra_new_cols]
    
    # Prepare DRP upload as TEXT columns
    upload_df = df_upload.copy()
    for c in upload_df.columns:
        upload_df[c] = upload_df[c].map(lambda x: None if pd.isna(x) else str(x))
        upload_df[c] = upload_df[c].astype(object)
    
    col_defs = []
    for c in upload_df.columns:
        col_name = str(c).replace('"', '""')
        col_defs.append(f'"{col_name}" TEXT')
    
    create_sql = f"""
    CREATE TABLE {target_new_fq}
    (
      {', '.join(col_defs)}
    )
    """
    
    with drp_conn:
        drp_conn.execute(f'DROP TABLE IF EXISTS {target_new_fq}')
        drp_conn.execute(create_sql)
        drp_conn.write(
            table=target_new_fq,
            df=upload_df,
            mode='append',
        )
    
        cnt_new_df = drp_conn.fetch(f"select count(*) as row_cnt from {target_new_fq}")
        new_cols_df = drp_conn.fetch(f"""
            select lower(column_name) as column_name
            from information_schema.columns
            where table_schema = '{schema_name}'
              and table_name = '{new_table_name}'
            order by ordinal_position
        """)

        # After DROP/CREATE privileges are gone — grant SELECT to Superset role
        grant_role = globals().get('drp_superset_grant_role', 'raisa_superset')
        try:
            drp_conn.execute(f'GRANT USAGE ON SCHEMA {schema_name} TO {grant_role}')
            drp_conn.execute(f'GRANT SELECT ON TABLE {target_new_fq} TO {grant_role}')
            print(f'OK: GRANT SELECT ON {target_new_fq} TO {grant_role}')
            grants_df = drp_conn.fetch(f"""
                select grantee, privilege_type
                from information_schema.role_table_grants
                where table_schema = '{schema_name}'
                  and table_name = '{new_table_name}'
                  and grantee = '{grant_role}'
            """)
            display(grants_df)
        except Exception as grant_exc:
            print('WARNING: GRANT for Superset failed:', type(grant_exc).__name__, str(grant_exc)[:400])
            print('Выполните вручную под владельцем таблицы:')
            print(f'  GRANT SELECT ON TABLE {target_new_fq} TO {grant_role};')
    
    new_cols = [str(x).strip().lower() for x in new_cols_df['column_name'].dropna().tolist()] if new_cols_df is not None and len(new_cols_df) else []
    missing_old_in_new = [c for c in old_cols if c not in new_cols]
    required_missing = [c for c in dashboard_required_cols if c not in new_cols]
    
    new_row_cnt = int(pd.to_numeric(cnt_new_df.iloc[0, 0], errors='coerce')) if cnt_new_df is not None and len(cnt_new_df) else 0
    
    # Basic KPI sanity (from uploaded dataframe before string cast)
    def _safe_sum(df, col):
        if col not in df.columns:
            return None
        return float(pd.to_numeric(df[col], errors='coerce').fillna(0).sum())
    
    kpi_sanity = pd.DataFrame([
        {'metric': 'row_cnt_new_table', 'value': new_row_cnt},
        {'metric': 'sum_retl_cnt', 'value': _safe_sum(df_upload, 'retl_cnt')},
        {'metric': 'sum_active_retl_cnt', 'value': _safe_sum(df_upload, 'active_retl_cnt')},
        {'metric': 'rows_active_retl_gt_retl', 'value': (
            float((
                pd.to_numeric(df_upload['active_retl_cnt'], errors='coerce').fillna(0)
                > pd.to_numeric(df_upload['retl_cnt'], errors='coerce').fillna(0)
            ).sum())
            if 'active_retl_cnt' in df_upload.columns and 'retl_cnt' in df_upload.columns
            else None
        )},
        {'metric': 'sum_term_cnt', 'value': _safe_sum(df_upload, 'term_cnt')},
        {'metric': 'sum_active_terms', 'value': _safe_sum(df_upload, 'active_terms')},
        {'metric': 'sum_commission_total', 'value': _safe_sum(df_upload, 'commission_total')},
        {'metric': 'sum_chod', 'value': _safe_sum(df_upload, 'chod')},
        {'metric': 'sum_fin_result', 'value': _safe_sum(df_upload, 'fin_result')},
        {'metric': 'sum_kedr_obshiy_chod_contrib', 'value': _safe_sum(df_upload, 'kedr_obshiy_chod_contrib')},
    ])
    
    compat_report = pd.DataFrame([
        {'check': 'old_columns_count', 'value': len(old_cols)},
        {'check': 'new_columns_count', 'value': len(new_cols)},
        {'check': 'missing_old_in_new_count', 'value': len(missing_old_in_new)},
        {'check': 'required_missing_count', 'value': len(required_missing)},
    ])
    
    print('Upload finished:', target_new_fq)
    print('Missing old columns in new:', missing_old_in_new[:20])
    print('Missing required dashboard columns:', required_missing)
    
    display(compat_report)
    display(kpi_sanity)
    
    display(pd.DataFrame({'missing_old_in_new': missing_old_in_new}))
    display(pd.DataFrame({'missing_required': required_missing}))


## DRP: выдать доступ Superset (`raisa_superset`)

Таблица уже залита. Запустите эту ячейку, если дашборд пишет `permission denied`.


In [ ]:
drp_user = input('DRP user: ').strip()
drp_password = getpass('DRP password: ')

drp = connect(
    to='DRP',
    user_params={
        'user_name': drp_user,
        'password': drp_password,
    },
)

with drp:
    drp.execute('GRANT USAGE ON SCHEMA sbx_da TO raisa_superset')
    drp.execute('GRANT SELECT ON TABLE sbx_da.tmp_shestopalov_acq_datamart_jan_jun TO raisa_superset')

print('OK: SELECT выдан raisa_superset на sbx_da.tmp_shestopalov_acq_datamart_jan_jun')


## Superset switch checklist (new table)

После upload в `sbx_da.tmp_shestopalov_acq_datamart_jan_jun`.

### Данные
1. Dataset на `sbx_da.tmp_shestopalov_acq_datamart_jan_jun`, temporal = `snapshot_month_start`. Sync columns (есть `kedr_obshiy_chod`, `kedr_obshiy_chod_contrib`).
2. Hard-refresh KPI / графики / сводную. В фильтре месяца должны быть **Jan–Jul** (июль — lake-only).
3. SQL Lab smoke: [`sources/sql/datamart_reload_smoke.sql`](../sources/sql/datamart_reload_smoke.sql) — есть `2026-07`.

### Сводная таблица — Общий ЧОД (Kedr)
1. Колонка/метрика: `MAX(kedr_obshiy_chod)` → label **Общий ЧОД**.
2. Если в totals сводной используется SUM по деньгам: для Kedr только `SUM(kedr_obshiy_chod_contrib)` (не `SUM(kedr_obshiy_chod)`).

### Распределение клиентов по сегментам ЧОД ТЭ
1. Dataset `vd_chod_clients_by_month`: SQL из [`sources/sql/vd_chod_clients_by_month.sql`](../sources/sql/vd_chod_clients_by_month.sql) → Sync columns.
2. Metrics: `COUNT DISTINCT agr_id_key`, `SUM(chod_sum)` (acquiring ЧОД ТЭ), `SUM(kedr_obshiy_chod_contrib)` → label **Общий ЧОД**.
3. Убрать `MAX(total_chod_placeholder)`.

### Фильтры
Chart ids сводной / 3×4 → [`advanced_json_extra_chart_ids.json`](../advanced_json_extra_chart_ids.json) → `kpiAndPivotChartIds` → `python tools/rebuild_advanced_json_scopes.py`.

Playbook: [`superset_update_after_v2_upload.md`](../superset_update_after_v2_upload.md).

### Retail points vs terminals
- Sync columns: убедись, что есть `active_retl_cnt`.
- Активные **торговые точки**: `SUM(active_retl_cnt)`.
- Неактивные ТТ: `SUM(retl_cnt) - SUM(active_retl_cnt)`.
- Активные **терминалы**: `SUM(active_terms)` (отдельный KPI).
- QC после upload: `sources/sql/active_retl_cnt_qc.sql`.
